# Note & Announcements

In [1]:
# @title Dependency Installation

import os
from time import  sleep

# Define a temporary marker file path
MARKER_FILE = '/tmp/colab_install_marker.txt'

# Only run the heavy installations if we haven't restarted yet
if not os.path.exists(MARKER_FILE):
    BAZEL_VERSION = '3.0.0'
    # Redirecting stdout and stderr to /dev/null for total silence
    !wget https://github.com/bazelbuild/bazel/releases/download/{BAZEL_VERSION}/bazel-{BAZEL_VERSION}-installer-linux-x86_64.sh --quiet
    !chmod +x bazel-{BAZEL_VERSION}-installer-linux-x86_64.sh
    !./bazel-{BAZEL_VERSION}-installer-linux-x86_64.sh > /dev/null 2>&1
    !sudo apt-get install python3-dev python3-setuptools git -qq -y > /dev/null 2>&1
    !git clone https://github.com/google/matched_markets -q > /dev/null 2>&1
    !python3 -m pip install ./matched_markets -q > /dev/null 2>&1
    !pip install colorama -q > /dev/null 2>&1
    !pip install gspread-dataframe -q > /dev/null 2>&1
    !pip install xlsxwriter -q > /dev/null 2>&1
    !pip install semopy -q > /dev/null 2>&1
    !pip install lingam -q > /dev/null 2>&1
    !pip install -U google-genai -q > /dev/null 2>&1
    !pip install groq  -q > /dev/null 2>&1
    !pip install quantile-forest -q > /dev/null 2>&1
    !pip install pytrends -q > /dev/null 2>&1
    !pip install tfcausalimpact  -q > /dev/null 2>&1

    # Create the marker file to remember we did this
    with open(MARKER_FILE, 'w') as f:
        f.write('installed')
    print("==================================================================================================")
    print("Packages Installed. Restart Session - Click Run All Again...")
    print("==================================================================================================")
    sleep(4)
    os.kill(os.getpid(), 9)

else:
    # If the marker file exists, we already restarted.
    # Clean it up so it works normally the next time you open the notebook.
    os.remove(MARKER_FILE)
    print("==================================================================================================")
    print("Session Already Restarted. Good to Go.")
    print("==================================================================================================")

Session Already Restarted. Good to Go.


# Geolift Code

In [2]:
# @title Import all necessary modules


print("============================================================================")
print("Import all necessary modules")
print("============================================================================")

"""Loading the necessary python modules."""
import altair as alt
import datetime
from datetime import timedelta
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import re
from scipy import stats

from IPython.display import display
from IPython.core.interactiveshell import InteractiveShell
from ipykernel.iostream import OutStream


import warnings
from colorama import Fore, Style

from matched_markets.methodology.common_classes import GeoAssignment
from matched_markets.methodology import geoeligibility
from matched_markets.methodology import tbrmmdata
from matched_markets.methodology import tbrmmdesignparameters
from matched_markets.methodology import tbrmmdiagnostics
from matched_markets.methodology import tbrmatchedmarkets
from matched_markets.methodology import tbrmmdesign
from matched_markets.methodology import utils
from matched_markets.methodology import tbr_iroas
from causalimpact import CausalImpact


import seaborn as sns
sns.set_theme(style="darkgrid")

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor
from quantile_forest import RandomForestQuantileRegressor
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_percentage_error,  mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
from sklearn.cluster import DBSCAN
import statsmodels.api as sm


import traceback
import gradio as gr

import json
import semopy
from semopy import Model
import tempfile
import os
import sys
import io
import contextlib
from PIL import Image
import ast
import traceback
import inspect

sns.set_theme()
warnings.filterwarnings('ignore')
InteractiveShell.ast_node_interactivity = "all"
from IPython.display import clear_output
from google.colab import output
import logging


import lingam

import google.generativeai as genai
from google.colab import userdata
from groq import Groq
from openai import OpenAI  # ✅ Swapped from Groq to OpenAI

clear_output()

In [3]:
# @title Core Functions and Model For Design Phase

def process_file_and_preview(file):
    if file is None:
        return [None] + [gr.update(choices=[], interactive=False)] * 4
    try:
        df = pd.read_csv(file.name) if file.name.endswith('.csv') else pd.read_excel(file.name)
        df = df.round(3)
        cols = df.columns.tolist()

        return (
            df.head(5),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True)
        )
    except Exception as e:
        print(f"Error: {e}")
        return [None] + [gr.update(choices=[], interactive=False)] * 4


def populate_date_choices(file, date_col):
    if file is None or not date_col:
        return gr.update(choices=[])
    try:
        df = pd.read_csv(file.name) if file.name.endswith('.csv') else pd.read_excel(file.name)
        # Extract unique dates and convert to string for the dropdown
        unique_dates = sorted(df[date_col].dropna().unique().astype(str).tolist())
        return gr.update(choices=unique_dates, interactive=True)
    except Exception as e:
        print(f"Error populating dates: {e}")
        return gr.update(choices=[], interactive=False)


def run_clustering(df, target_col, cost_col, additional_cols, date_col, geo_col, exclude_dates, weight_str):
    additional_cols = additional_cols or []

    # 1. ❌ Removed 'file' check, replaced with 'df'
    if df is None or not target_col or not cost_col or not geo_col:
      return "⚠️ Please upload a dataset and select all required columns.", None, *([gr.update(visible=False)] * 11), None

    try:
        # 2. 🧠 PARSE SEM WEIGHTS
        weight_map = {}
        if weight_str and weight_str.strip() != "":
            try:
                weight_map = ast.literal_eval(weight_str)
            except Exception as e:
                raise ValueError(f"Invalid SEM Weights format: {e}")

        # FILTER BEFORE CLUSTERING
        if exclude_dates:
            df = df[~df[date_col].astype(str).isin(exclude_dates)]

        # 1. DYNAMIC FEATURE AGGREGATION
        agg_dict = {
            'avg_revenue': (target_col, 'mean'),
            'avg_cost': (cost_col, 'mean')
        }

        for col in additional_cols:
            if col not in [target_col, cost_col]: # SAFETY CHECK
                agg_dict[f'avg_{col}'] = (col, 'mean')

        geo_features = df.groupby(geo_col).agg(**agg_dict).reset_index().dropna()

        # =====================================================================
        # 2. DYNAMIC SCALING & FEATURE WEIGHTING (COMBINED)
        # =====================================================================
        scaler = StandardScaler()
        features_to_scale = list(agg_dict.keys())
        scaled_data = scaler.fit_transform(geo_features[features_to_scale])

        scaled_df = pd.DataFrame(scaled_data, columns=features_to_scale)

        # Map the specific cost column to 'avg_cost' to match agg_dict keys safely
        weight_map_copy = weight_map.copy()
        actual_cost_key = f'avg_{cost_col}'
        if actual_cost_key in weight_map_copy:
            weight_map_copy['avg_cost'] = weight_map_copy.pop(actual_cost_key)

        # Apply the SEM weights (Default to 1.0 if a feature wasn't in the causal model)
        for col in features_to_scale:
            weight = weight_map_copy.get(col, 1.0)
            scaled_df[col] = scaled_df[col] * weight

        weighted_scaled_data = scaled_df.values

        # ========================================================================================================
        # 3. DBSCAN CLUSTERING (Replaces K-Means and Elbow Method Instead Automatically finds K, tags Noise as -1)
        # eps = 0.8: Maximum distance between two markets to be considered neighbors
        # min_samples = 3: Minimum markets required to form a valid match cohort
        # ========================================================================================================
        dbscan = DBSCAN(eps=0.10, min_samples=10, metric='correlation')

        geo_features['cluster'] = dbscan.fit_predict(weighted_scaled_data)

        # 4. LABELING & SORTING (CRITICAL FIX FOR run_match_market COMPATIBILITY)
        # We must keep geo_features['cluster'] as an INTEGER for your next function.
        valid_mask = geo_features['cluster'] != -1
        mapping = {-1: -1} # Noise stays as integer -1

        if valid_mask.any():
            sorted_valid = geo_features[valid_mask].groupby('cluster')['avg_revenue'].mean().sort_values().index
            mapping.update({old_id: new_id for new_id, old_id in enumerate(sorted_valid)})

        # Map to the new sorted integers
        geo_features['cluster'] = geo_features['cluster'].map(mapping)

        # Build dropdown choices that won't break the int(c.split()[-1]) logic
        unique_clusters = sorted(geo_features['cluster'].unique().tolist())
        cluster_choices = []
        for c in unique_clusters:
            if c == -1:
                cluster_choices.append("Outliers -1") # split()[-1] becomes "-1", which parses safely to integer!
            else:
                cluster_choices.append(f"Cluster {c}")

        # Merge the integer cluster column back to df
        df_clustered = pd.merge(df, geo_features[[geo_col, 'cluster']], on=geo_col, how='left')

        # =====================================================================
        # 5. DYNAMIC VISUALIZATION (THE ULTIMATE BUBBLE CHART)
        # =====================================================================

        # Create a string label JUST for the chart so it looks nice to the user
        geo_features['Cluster_Label'] = geo_features['cluster'].apply(
            lambda x: "Noise (Outliers)" if x == -1 else f"Cluster {x}"
        )

        x_axis_label = f'Avg {cost_col}' if "cost" not in cost_col.lower() and "spend" not in cost_col.lower() else f'Avg {cost_col} ($)'

        # 5A. Build the Hover Dictionary
        hover_data_dict = {
            'Cluster_Label': True,
            'avg_revenue': ':.2f',
            'avg_cost': ':.2f',
            'cluster': False # Hide the raw int from the tooltip
        }

        for col in additional_cols:
            if col not in [target_col, cost_col]:
                hover_data_dict[f'avg_{col}'] = ':.2f'

        # 5B. Determine Bubble Size
        size_variable = None
        if additional_cols:
            first_extra = additional_cols[0]
            if first_extra not in [target_col, cost_col]:
                size_variable = f'avg_{first_extra}'

        # 5C. Generate the Scatter Plot
        fig = px.scatter(
            geo_features,
            x='avg_revenue',
            y='avg_cost',
            size=size_variable,
            color='Cluster_Label',
            hover_name=geo_col,
            hover_data=hover_data_dict,
            title=f"Market Cohorts (DBSCAN) | Hover for Geo Details",
            labels={'avg_revenue': 'Avg Revenue', 'avg_cost': x_axis_label, 'Cluster_Label': 'Cohort'},
            template="plotly_white",
            color_discrete_map={"Noise (Outliers)": "rgba(200, 200, 200, 0.6)"}
        )

        if size_variable:
            fig.update_traces(marker=dict(sizemin=5, sizemode='area'))
        else:
            fig.update_traces(marker=dict(size=12))

        fig.update_layout(xaxis_tickformat='$,.0f', yaxis_tickformat='$,.0f')
        # =====================================================================

        noise_count = len(geo_features[geo_features['cluster'] == -1])
        valid_count = len(geo_features) - noise_count
        num_clusters = len([c for c in unique_clusters if c != -1])


        status_msg = f"✅ Analysis Complete! Grouped {valid_count} markets into {num_clusters} valid clusters. Automatically isolated {noise_count} noisy outliers."

        return (
            status_msg, fig,
            gr.update(choices=cluster_choices, visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            gr.update(visible=False, interactive=False),
            gr.update(visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            df_clustered
        )

    except Exception as e:
        return f"❌ Error: {str(e)}", None, *([gr.update(visible=False)] * 11), None


def update_geo_dropdowns(df_clustered, cluster_selection, geo_col):
    if df_clustered is None or not cluster_selection or not geo_col:
        return gr.update(choices=[]), gr.update(choices=[]), gr.update(choices=[])

    try:
        # 1. Grab the cluster IDs the user selected
        selected_clusters = [int(c.split()[-1]) for c in cluster_selection]

        # 2. Filter the dataset to ONLY include those specific clusters
        filtered_df = df_clustered[df_clustered['cluster'].isin(selected_clusters)]

        # 3. Clean the geos: Drop NaNs and force them to strings so sorted() doesn't crash!
        valid_geos = sorted(filtered_df[geo_col].dropna().astype(str).unique().tolist())

        # 4. Push this exact list to all three of your dropdowns
        return gr.update(choices=valid_geos), gr.update(choices=valid_geos), gr.update(choices=valid_geos)

    except Exception as e:
        print(f"Error updating geo dropdowns: {e}")
        return gr.update(choices=[]), gr.update(choices=[]), gr.update(choices=[])


def run_match_market(df_clustered, cluster_selection, target_col, geo_col, date_col, target_type, test_direction, iroas_or_cpa, exp_length, budget, budget_cut, req_geo, excl_geo, excl_comp):
    if df_clustered is None or df_clustered.empty:
        return "⚠️ No clustered data found.", None, None, None, None, gr.update(visible=False), gr.update(visible=False)
    if not cluster_selection:
        return "⚠️ Select at least one cluster.", None, None, None, None, gr.update(visible=False), gr.update(visible=False)

    try:
        selected_clusters = [int(c.split()[-1]) for c in cluster_selection]
        optimal_testing_data = df_clustered[df_clustered['cluster'].isin(selected_clusters)].copy()
        total_sweet_spot_geos = optimal_testing_data[geo_col].nunique()

        if target_type == "Revenue":
            minimum_detectable_iROAS = float(iroas_or_cpa)
        else:
            minimum_detectable_iROAS = 1.0 / float(iroas_or_cpa)

        experiment_duration = int(exp_length)
        n_pretest = len(optimal_testing_data[date_col].unique())

        day_week_exclude = []
        periods_to_exclude = utils.find_days_to_exclude(day_week_exclude)
        days_exclude = utils.expand_time_windows(periods_to_exclude)

        confidence_level = 0.90
        power_level = 0.80

        tbr_parameters = tbrmmdesignparameters.TBRMMDesignParameters(
            n_test=experiment_duration,
            iroas=minimum_detectable_iROAS,
            volume_ratio_tolerance=np.inf,
            geo_ratio_tolerance=np.inf,
            treatment_share_range=(0.0001, 0.9999),
            budget_range=(0.1, float(budget)),
            treatment_geos_range=(3, 20),
            control_geos_range=(1, total_sweet_spot_geos - 1),
            n_geos_max=total_sweet_spot_geos,
            n_pretest_max=n_pretest,
            n_designs=30,
            sig_level=confidence_level,
            power_level=power_level,
            min_corr=0.8,
            rho_max=0.995,
            flevel=0.9
        )

        geo_mapping = []
        for city in optimal_testing_data[geo_col].unique():
            is_treat, is_control, is_excluded = True, True, False
            if city in excl_comp:
                is_treat, is_control, is_excluded = False, False, True
            elif city in req_geo:
                is_treat, is_control, is_excluded = True, False, False
            elif city in excl_geo:
                is_treat, is_control, is_excluded = False, True, False
            geo_mapping.append({'geo': city, 'control': is_control, 'treatment': is_treat, 'exclude': is_excluded})

        custom_eligibility_df = pd.DataFrame(geo_mapping)
        custom_eligibility_obj = geoeligibility.GeoEligibility(custom_eligibility_df)

        data_for_design = optimal_testing_data[~optimal_testing_data[date_col].isin(days_exclude)].copy()
        formatted_mm_df = data_for_design.rename(columns={
            date_col: 'date',
            geo_col: 'geo',
            target_col: 'response'
        })

        tbrclass = tbrmmdata.TBRMMData(df=formatted_mm_df, response_column='response', geo_eligibility=custom_eligibility_obj)
        MMclass = tbrmatchedmarkets.TBRMatchedMarkets(data=tbrclass, parameters=tbr_parameters)
        matched_designs = MMclass.greedy_search()

        if len(matched_designs) == 0:
            raise ValueError("❌ No valid designs found. Try lowering min_corr or adjusting parameters.")

        matched_designs.sort(reverse=True)

        pseudo_test_duration = experiment_duration
        optimal_testing_data[date_col] = pd.to_datetime(optimal_testing_data[date_col], format='mixed', dayfirst=True)
        latest_date = optimal_testing_data[date_col].max()
        split_date = latest_date - timedelta(days=pseudo_test_duration)

        train_data = optimal_testing_data[optimal_testing_data[date_col] <= split_date].copy()
        pseudo_test_data = optimal_testing_data[optimal_testing_data[date_col] > split_date].copy()

        planned_budget_change = float(budget_cut)
        if target_type == "Revenue":
            expected_impact_absolute = planned_budget_change * float(iroas_or_cpa)
        else:
            expected_impact_absolute = planned_budget_change / float(iroas_or_cpa)

        if test_direction == "Hold-out":
            expected_impact_absolute = -abs(expected_impact_absolute)
        else:
            expected_impact_absolute = abs(expected_impact_absolute)

        geolift_table = []
        geolist_table = []
        top_n = len(matched_designs)

        for i in range(top_n):
            design = matched_designs[i]
            treat_geos = list(design.treatment_geos)
            ctrl_geos = list(design.control_geos)
            correlation = design.diag.corr
            location_str = ", ".join(treat_geos)
            location_cntrl_str = ", ".join(ctrl_geos)
            match_name = f"Match {i + 1}"

            try: aa_test_status = '✅ Pass' if design.score.score.aa_test else '❌ Fail'
            except AttributeError: aa_test_status = '❓ N/A'

            try: bb_test_status = '✅ Pass' if design.score.score.bb_test else '❌ Fail'
            except AttributeError: bb_test_status = '❓ N/A'

            dw_test_status = '❓ N/A'
            if hasattr(design.score, 'dw_test'): dw_test_status = '✅ Pass' if design.score.dw_test else '❌ Fail'
            elif hasattr(design.score, 'score') and hasattr(design.score.score, 'dw_test'): dw_test_status = '✅ Pass' if design.score.score.dw_test else '❌ Fail'
            elif hasattr(design, 'diag') and hasattr(design.diag, 'dw_test'): dw_test_status = '✅ Pass' if design.diag.dw_test else '❌ Fail'

            if '✅' in aa_test_status and '✅' in bb_test_status and '✅' in dw_test_status:
                overall_status = '🏆 Valid'
            else:
                overall_status = '⚠️ Invalid'

            test_sim = pseudo_test_data.copy()
            treatment_mask = test_sim[geo_col].isin(treat_geos)

            test_baseline_revenue = pseudo_test_data[pseudo_test_data[geo_col].isin(treat_geos)][target_col].sum()
            train_baseline_revenue = train_data[train_data[geo_col].isin(treat_geos)][target_col].sum()

            if test_baseline_revenue > 0:
                dynamic_lift_pct = expected_impact_absolute / test_baseline_revenue
            else:
                dynamic_lift_pct = 0

            test_sim.loc[treatment_mask, target_col] = test_sim.loc[treatment_mask, target_col] * (1 + dynamic_lift_pct)

            simulated_lift_dollars = expected_impact_absolute

            control_pool_pre = train_data[train_data[geo_col].isin(ctrl_geos)][target_col].sum()
            control_pool_post = test_sim[test_sim[geo_col].isin(ctrl_geos)][target_col].sum()
            treat_pre = train_data[train_data[geo_col].isin(treat_geos)][target_col].sum()
            treat_post_injected = test_sim[test_sim[geo_col].isin(treat_geos)][target_col].sum()

            market_drift = control_pool_post / control_pool_pre if control_pool_pre > 0 else 1
            counterfactual_baseline_revenue = treat_pre * market_drift
            estimated_lift_dollars = treat_post_injected - counterfactual_baseline_revenue

            absolute_error_pct = abs(estimated_lift_dollars - simulated_lift_dollars) / abs(simulated_lift_dollars) if simulated_lift_dollars != 0 else 0

            geolift_table.append({
                "Match No": match_name,
                "Status": overall_status,
                "Location": location_str,
                "Correlation": correlation,
                "AA Test": aa_test_status,
                "Brownian Bridge": bb_test_status,
                "Durbin-Watson": dw_test_status,
                "Train Baseline Revenue": train_baseline_revenue,
                "Estimated Lift": estimated_lift_dollars,
                "Absolute % error": absolute_error_pct * 100
            })

            geolist_table.append({
                "Match No": match_name,
                "Status": overall_status,
                "Treatment": location_str,
                "Control": location_cntrl_str
            })

        df_geolist = pd.DataFrame(geolist_table)
        df_geolift = pd.DataFrame(geolift_table)

        df_geolift = df_geolift.sort_values('Absolute % error', ascending=True).reset_index(drop=True)
        df_geolift.index += 1

        formatted_df = df_geolift.copy()
        target_prefix = "$" if target_type == "Revenue" else ""
        target_label = "Revenue" if target_type == "Revenue" else target_type

        formatted_df = formatted_df.rename(columns={
            "Train Baseline Revenue": f"Pre-Test Baseline {target_label}",
            "Estimated Lift": f"Estimated Synthetic Lift {target_label}",
            "Absolute % error": f"Lift Estimation Error (%)",
        })

        formatted_df['Correlation'] = formatted_df['Correlation'].apply(lambda x: f"{x:.4f}")
        formatted_df[f"Pre-Test Baseline {target_label}"] = formatted_df[f"Pre-Test Baseline {target_label}"].apply(lambda x: f"{target_prefix}{x:,.0f}")
        formatted_df[f"Estimated Synthetic Lift {target_label}"] = formatted_df[f"Estimated Synthetic Lift {target_label}"].apply(lambda x: f"{target_prefix}{x:,.0f}")
        formatted_df['Lift Estimation Error (%)'] = formatted_df['Lift Estimation Error (%)'].round(2).astype(str) + "%"

        status_msg = f"✅ Match Market Search Complete! Evaluated {top_n} designs."
        match_choices = [f"Match {i + 1}" for i in range(top_n)]

        formatted_df_view = formatted_df.drop(columns=[
            f"Estimated Synthetic Lift {target_label}",
            "Lift Estimation Error (%)"
        ])

        # 📝 Returns exactly the 7 items Gradio expects!
        return (
            status_msg,
            formatted_df,
            df_geolist,
            matched_designs,
            optimal_testing_data,
            gr.update(choices=match_choices, visible=True, interactive=True),
            gr.update(visible=True, interactive=True),
            formatted_df_view
        )

    except Exception as e:
        full_error = traceback.format_exc()
        error_markdown = f"### ❌ Match Market Failed\n**Error Details:**\n```python\n{full_error}\n```"
        error_df = pd.DataFrame({"Status": ["Failed"], "Message": ["Check the error log above."]})
        return error_markdown, error_df, None, None, None, gr.update(visible=False), gr.update(visible=False)


def model_selection(tbr_abs_err_pct, qrf_abs_err_pct, bayes_abs_err_pct):
  errors = {
        "Frequentist DiD": tbr_abs_err_pct, # TBR Error
        "Machine Learning QRF": qrf_abs_err_pct,
        "Bayesian STS (CausalImpact)": bayes_abs_err_pct
    }

  champion_name = min(errors, key=errors.get)
  champion_error = errors[champion_name]

  return champion_name

def chart_selection(champion_name, target_type, post_period, inf_df, did_daily_df, qrf_daily_df, qrf_cv_preds, actual_y, intervention_date_bsts):
    """
    Selects and builds the correct counterfactual and cumulative subplots based on the champion model.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd

    # 1. Clear memory to prevent Gradio UI memory leaks
    plt.clf()
    plt.close('all')
    sns.set_theme(style="darkgrid")

    # ---------------------------------------------------------
    # A. BAYESIAN STS (CAUSALIMPACT)
    # ---------------------------------------------------------
    if champion_name == "Bayesian STS (CausalImpact)":
        try:
            dates = pd.to_datetime(inf_df.index)
            pred_col = 'complete_preds_means'
            lower_col = 'complete_preds_lower'
            upper_col = 'complete_preds_upper'
            cum_col = 'post_cum_effects_means'
            cum_lower = 'post_cum_effects_lower'
            cum_upper = 'post_cum_effects_upper'

            cum_means = inf_df[cum_col].fillna(0)
            cum_lower_bounds = inf_df[cum_lower].fillna(0)
            cum_upper_bounds = inf_df[cum_upper].fillna(0)

            fig_ci, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 9), sharex=True)

            # --- ROW 1: ACTUAL VS PREDICTED ---
            ax1.fill_between(dates, inf_df[lower_col], inf_df[upper_col], color='#3498DB', alpha=0.2, label='90% CI')
            ax1.plot(dates, inf_df[pred_col], color='#3498DB', linestyle='--', linewidth=2, label='Predicted Counterfactual')
            ax1.plot(dates, actual_y, color='#2C3E50', linewidth=2.5, label=f'Actual {target_type}')

            ax1.set_title(f'CausalImpact: Actual vs. Predicted {target_type}', fontsize=14, fontweight='bold')
            ax1.set_ylabel(target_type, fontsize=12)
            ax1.legend(loc='upper left')

            # --- ROW 2: CUMULATIVE EFFECT ---
            ax2.fill_between(dates, cum_lower_bounds, cum_upper_bounds, color='#E74C3C', alpha=0.2, label='90% CI')
            ax2.plot(dates, cum_means, color='#E74C3C', linestyle='--', linewidth=2.5, label='Cumulative Effect')
            ax2.axhline(0, color='black', linewidth=1.5, alpha=0.6)

            ax2.set_title('Cumulative Incremental Effect', fontsize=14, fontweight='bold')
            ax2.set_ylabel(f'BSTS: Cumulative Impact', fontsize=12)
            ax2.set_xlabel('Date', fontsize=12)
            ax2.legend(loc='upper left')

            for ax in [ax1, ax2]:
                ax.axvline(x=intervention_date_bsts, color='#7F8C8D', linestyle='--', linewidth=2.5, zorder=0)
                ax.text(intervention_date_bsts, ax.get_ylim()[1] * 0.95, '   Test Launched',
                        color='#7F8C8D', ha='left', va='top', fontsize=11, fontweight='bold')

            fig_ci.autofmt_xdate(rotation=45)
            fig_ci.tight_layout()

            return fig_ci

        except Exception as e:
            print(f"Error generating Seaborn CausalImpact plot: {e}")
            return plt.figure()

    # ---------------------------------------------------------
    # B. FREQUENTIST DID (TBR)
    # ---------------------------------------------------------
    elif champion_name == "Frequentist DiD":
        try:
            did_daily_copy = did_daily_df.copy()
            did_daily_copy['date'] = pd.to_datetime(did_daily_copy['date'])
            intervention_date_tbr = pd.to_datetime(post_period[0])
            is_post = did_daily_copy['date'] >= intervention_date_tbr

            # FIX: Use the 'actual' column that already has the signal injected from the parent function
            did_daily_copy['bsts_actual'] = did_daily_copy['actual']

            did_daily_copy['actual'] = did_daily_copy['actual'].clip(lower=0)
            did_daily_copy['predicted'] = did_daily_copy['predicted'].clip(lower=0)
            did_daily_copy['lower'] = did_daily_copy['lower'].clip(lower=0)
            did_daily_copy['bsts_actual'] = did_daily_copy['bsts_actual'].clip(lower=0)

            # Cumulative calculations
            did_daily_copy['point_effect'] = np.where(is_post, did_daily_copy['bsts_actual'] - did_daily_copy['predicted'], 0)
            did_daily_copy['cum_effect'] = did_daily_copy['point_effect'].cumsum()
            did_daily_copy['point_moe'] = np.where(is_post, did_daily_copy['upper'] - did_daily_copy['predicted'], 0)
            did_daily_copy['cum_moe'] = did_daily_copy['point_moe'].cumsum()
            did_daily_copy['cum_lower'] = did_daily_copy['cum_effect'] - did_daily_copy['cum_moe']
            did_daily_copy['cum_upper'] = did_daily_copy['cum_effect'] + did_daily_copy['cum_moe']

            fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 9), sharex=True)

            axes[0].fill_between(did_daily_copy['date'], did_daily_copy['lower'], did_daily_copy['upper'], color='#3498DB', alpha=0.2, label='90% CI')
            axes[0].plot(did_daily_copy['date'], did_daily_copy['predicted'], color='#3498DB', linestyle='--', linewidth=2, label='TBR Counterfactual')
            axes[0].plot(did_daily_copy['date'], did_daily_copy['bsts_actual'], color='#2C3E50', linewidth=2.5, label=f'Actual {target_type}')
            axes[0].set_title(f'TBR: Actual vs. Predicted {target_type}', fontsize=14, fontweight='bold')
            axes[0].set_ylabel(target_type, fontsize=12)
            axes[0].legend(loc='upper left')

            axes[1].fill_between(did_daily_copy['date'], did_daily_copy['cum_lower'], did_daily_copy['cum_upper'], color='#E74C3C', alpha=0.2, label='90% CI')
            axes[1].plot(did_daily_copy['date'], did_daily_copy['cum_effect'], color='#E74C3C', linestyle='--', linewidth=2.5, label='Cumulative Effect')
            axes[1].axhline(0, color='black', linewidth=1.5, alpha=0.6)
            axes[1].set_title('Cumulative Incremental Effect', fontsize=14, fontweight='bold')
            axes[1].set_ylabel('TBR: Cumulative Impact', fontsize=12)
            axes[1].set_xlabel('Date', fontsize=12)
            axes[1].legend(loc='upper left')

            for ax in axes:
                ax.axvline(x=intervention_date_bsts, color='#7F8C8D', linestyle='--', linewidth=2.5, zorder=0)
                ax.text(intervention_date_bsts, ax.get_ylim()[1] * 0.95, '   Test Launched',
                        color='#7F8C8D', ha='left', va='top', fontsize=11, fontweight='bold')

            fig.autofmt_xdate(rotation=45)
            fig.tight_layout()
            return fig

        except Exception as e:
            print(f"Error generating DiD/TBR plot: {e}")
            return plt.figure()

    # ---------------------------------------------------------
    # C. MACHINE LEARNING QRF
    # ---------------------------------------------------------
    elif champion_name == "Machine Learning QRF":
        try:
            qrf_daily_copy = qrf_daily_df.copy()
            qrf_daily_copy['date'] = pd.to_datetime(qrf_daily_copy['date'])
            intervention_date_qrf = pd.to_datetime(post_period[0])
            is_post_qrf = qrf_daily_copy['date'] >= intervention_date_qrf

            # FIX: Safely assign cross-validation predictions by exact length rather than date logic
            num_pre_days = len(qrf_cv_preds)
            qrf_daily_copy.iloc[:num_pre_days, qrf_daily_copy.columns.get_loc('predicted')] = qrf_cv_preds

            # FIX: Use the 'actual' column that already has the signal injected from the parent function
            qrf_daily_copy['bsts_actual'] = qrf_daily_copy['actual']

            qrf_daily_copy['actual'] = qrf_daily_copy['actual'].clip(lower=0)
            qrf_daily_copy['predicted'] = qrf_daily_copy['predicted'].clip(lower=0)
            qrf_daily_copy['lower'] = qrf_daily_copy['lower'].clip(lower=0)
            qrf_daily_copy['bsts_actual'] = qrf_daily_copy['bsts_actual'].clip(lower=0)

            qrf_daily_copy['point_effect'] = np.where(is_post_qrf, qrf_daily_copy['bsts_actual'] - qrf_daily_copy['predicted'], 0)
            qrf_daily_copy['cum_effect'] = qrf_daily_copy['point_effect'].cumsum()

            # FIX: Fixed typo here! Changed qrf_daily_df to qrf_daily_copy so it doesn't throw a KeyError
            qrf_daily_copy['point_moe'] = np.where(is_post_qrf, qrf_daily_copy['upper'] - qrf_daily_copy['predicted'], 0)
            qrf_daily_copy['cum_moe'] = np.sqrt((qrf_daily_copy['point_moe']**2).cumsum())

            qrf_daily_copy['cum_lower'] = qrf_daily_copy['cum_effect'] - qrf_daily_copy['cum_moe']
            qrf_daily_copy['cum_upper'] = qrf_daily_copy['cum_effect'] + qrf_daily_copy['cum_moe']

            fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 9), sharex=True)

            axes[0].fill_between(qrf_daily_copy['date'], qrf_daily_copy['lower'], qrf_daily_copy['upper'], color='#3498DB', alpha=0.2, label='90% CI')
            axes[0].plot(qrf_daily_copy['date'], qrf_daily_copy['predicted'], color='#3498DB', linestyle='--', linewidth=2, label='QRF Counterfactual')
            axes[0].plot(qrf_daily_copy['date'], qrf_daily_copy['bsts_actual'], color='#2C3E50', linewidth=2.5, label=f'Actual {target_type}')
            axes[0].set_title(f'Machine Learning QRF: Actual vs. Predicted {target_type}', fontsize=14, fontweight='bold')
            axes[0].set_ylabel(target_type, fontsize=12)
            axes[0].legend(loc='upper left')

            axes[1].fill_between(qrf_daily_copy['date'], qrf_daily_copy['cum_lower'], qrf_daily_copy['cum_upper'], color='#E74C3C', alpha=0.2, label='90% CI')
            axes[1].plot(qrf_daily_copy['date'], qrf_daily_copy['cum_effect'], color='#E74C3C', linestyle='--', linewidth=2.5, label='Cumulative Effect')
            axes[1].axhline(0, color='black', linewidth=1.5, alpha=0.6)
            axes[1].set_title('Cumulative Incremental Effect', fontsize=14, fontweight='bold')
            axes[1].set_ylabel('QRF: Cumulative Impact', fontsize=12)
            axes[1].set_xlabel('Date', fontsize=12)
            axes[1].legend(loc='upper left')

            for ax in axes:
                ax.axvline(x=intervention_date_bsts, color='#7F8C8D', linestyle='--', linewidth=2.5, zorder=0)
                ax.text(intervention_date_bsts, ax.get_ylim()[1] * 0.95, '   Test Launched',
                        color='#7F8C8D', ha='left', va='top', fontsize=11, fontweight='bold')

            fig.autofmt_xdate(rotation=45)
            fig.tight_layout()
            return fig

        except Exception as e:
            print(f"Error generating QRF plot: {e}")
            return plt.figure()

    return plt.figure()

def design_summary_choosen_design(match_selection, matched_designs, optimal_testing_data, experiment_duration, target_column, geo_column, date_column, cost_column, minimum_detectable_iROAS, test_direction, target_type, budget_input, formatted_df):
    try:
        # --- UI Safety Checks ---
        if not matched_designs or optimal_testing_data is None or not match_selection:
            return gr.update(value="⚠️ Please run the Match Market Analysis first and select a design.", visible=True)

        try:
            selected_index = int(match_selection.split()[-1]) - 1
        except (ValueError, AttributeError, IndexError):
            return gr.update(value="⚠️ Invalid match selection format.", visible=True)

        if selected_index < 0 or selected_index >= len(matched_designs):
            return gr.update(value="⚠️ Match number out of bounds.", visible=True)

        alpha_val = 0.1
        chosen_design = matched_designs[selected_index]
        treatment_geos = list(chosen_design.treatment_geos)
        control_geos = list(chosen_design.control_geos)
        iroas_or_cpa = minimum_detectable_iROAS

        # =============================================================================
        # QRF & DATA PREP
        # =============================================================================
        ml_data = optimal_testing_data.copy()
        ml_data[date_column] = pd.to_datetime(ml_data[date_column], format='mixed', dayfirst=True)

        treat_df = ml_data[ml_data[geo_column].isin(treatment_geos)].groupby(date_column)[target_column].sum().reset_index().rename(columns={target_column: 'y_treat'})
        ctrl_df = ml_data[ml_data[geo_column].isin(control_geos)].groupby(date_column)[target_column].sum().reset_index().rename(columns={target_column: 'X_ctrl'})

        model_df = pd.merge(treat_df, ctrl_df, on=date_column, how='inner').sort_values(date_column)
        model_df['day_of_week'] = model_df[date_column].dt.dayofweek

        split_index = len(model_df) - int(experiment_duration)
        train_df = model_df.iloc[:split_index]
        test_df = model_df.iloc[split_index:]

        X_train = train_df[['X_ctrl', 'day_of_week']]
        y_train = train_df['y_treat']
        X_test = test_df[['X_ctrl', 'day_of_week']]
        y_actual = test_df['y_treat']

        qrf = RandomForestQuantileRegressor(n_estimators=100, random_state=42)
        qrf.fit(X_train, y_train)

        qrf_cv_preds = cross_val_predict(qrf, X_train, y_train, cv=5)
        if len(qrf_cv_preds.shape) > 1: qrf_cv_preds = qrf_cv_preds[:, 1]

        qrf_rmse = np.sqrt(mean_squared_error(y_train, qrf_cv_preds))
        qrf_r2 = r2_score(y_train, qrf_cv_preds)

        y_pred_quantiles = qrf.predict(X_test, quantiles=[0.05, 0.5, 0.95])
        y_pred_lower  = y_pred_quantiles[:, 0]
        y_pred_median = y_pred_quantiles[:, 1]
        y_pred_upper  = y_pred_quantiles[:, 2]

        total_actual = y_actual.sum()
        total_predicted = y_pred_median.sum()
        total_lower_bound = y_pred_lower.sum()
        total_upper_bound = y_pred_upper.sum()

        difference = total_actual - total_predicted
        abs_pct = (abs(difference) / total_actual) * 100 if total_actual > 0 else 0
        qrf_margin_of_error = total_predicted - total_lower_bound
        min_impact = chosen_design.diag.estimate_required_impact(chosen_design.diag.corr)

        target_roas = float(iroas_or_cpa)
        if test_direction == "Hold-out":
            simulated_ground_truth = -abs(float(budget_input) * target_roas)
        else:
            simulated_ground_truth = abs(float(budget_input) * target_roas)

        total_actual_injected = total_actual + simulated_ground_truth
        qrf_estimated_lift = total_actual_injected - total_predicted

        qrf_absolute_error_pct = (abs(qrf_estimated_lift - simulated_ground_truth) / abs(simulated_ground_truth)) * 100 if simulated_ground_truth != 0 else 0
        lower_bound_lift = simulated_ground_truth - qrf_margin_of_error
        upper_bound_lift = simulated_ground_truth + qrf_margin_of_error

        # 1. Predict across the ENTIRE timeline (Pre + Post) to draw a complete chart
        X_all = model_df[['X_ctrl', 'day_of_week']]
        all_quantiles = qrf.predict(X_all, quantiles=[0.05, 0.5, 0.95])

        # 2. Build the baseline daily dataframe
        qrf_daily_df = pd.DataFrame({
            'date': model_df[date_column],
            'actual': model_df['y_treat'], # Base actuals
            'predicted': all_quantiles[:, 1],
            'lower': all_quantiles[:, 0],
            'upper': all_quantiles[:, 2]
        })

        # 3. Inject the synthetic signal into the post-period Actuals (for the Plot)
        # We distribute the total ground truth evenly across the test days
        n_post_days = len(test_df)
        daily_injection = simulated_ground_truth / n_post_days if n_post_days > 0 else 0

        # Identify post-period rows and add the injected lift
        post_start_date = test_df.iloc[0][date_column]
        is_post = qrf_daily_df['date'] >= post_start_date

        qrf_daily_df.loc[is_post, 'actual'] = qrf_daily_df.loc[is_post, 'actual'] + daily_injection

        # =============================================================================
        # TBR ANALYSIS
        # =============================================================================
        tbr_weight = y_train.sum() / X_train['X_ctrl'].sum()
        tbr_train_preds = X_train['X_ctrl'] * tbr_weight
        tbr_rmse = np.sqrt(mean_squared_error(y_train, tbr_train_preds))
        tbr_r2 = r2_score(y_train, tbr_train_preds)

        df_analysis = ml_data.copy()
        def get_assignment(geo_name):
            if geo_name in treatment_geos: return 1
            elif geo_name in control_geos: return 0
            else: return np.nan

        df_analysis['assignment'] = df_analysis[geo_column].apply(get_assignment)
        df_analysis = df_analysis.dropna(subset=['assignment'])

        # ✅ THE CORRECTED CODE: Forces the split based on experiment_duration
        # 0 = Pre-test (Train), 1 = Test (Experiment)
        last_date_tbr = pd.to_datetime(df_analysis[date_column], format='mixed', dayfirst=True).max()
        test_start_date = last_date_tbr - pd.Timedelta(days=int(experiment_duration))

        df_analysis['period'] = df_analysis[date_column].apply(
            lambda x: 1 if x > test_start_date else 0
        )

        # Rename columns to match the TBR library's expected schema
        geox_data = df_analysis.rename(columns={date_column: 'date', geo_column: 'geo', target_column: 'response', cost_column: 'cost'})

        # Run the Time-Based Regression model
        tbr_model = tbr_iroas.TBRiROAS(use_cooldown=False)
        tbr_model.fit(geox_data, key_group='assignment', group_control=0, group_treatment=1)

        # ✅ FIX: Explicitly generate the summary data frame here at an 80% confidence level
        tbr_summary_df = tbr_model.summary(level=0.80, tails=2)

        # ✅ Extract the point estimate value cleanly
        tbr_point_estimate = tbr_summary_df['incremental_response'].values[0]

        # Isolate the most recent data (the simulated test period)
        last_date = pd.to_datetime(ml_data[date_column], format='mixed', dayfirst=True).max()
        start_date = last_date - pd.Timedelta(days=int(experiment_duration))
        recent_data = ml_data[ml_data[date_column] > start_date]
        baseline_revenue = recent_data[recent_data[geo_column].isin(treatment_geos)][target_column].sum()
        current_brand_spend = recent_data[recent_data[geo_column].isin(treatment_geos)][cost_column].sum()

        optimal_budget = min_impact / target_roas if target_roas > 0 else 0
        additional_budget_needed = optimal_budget - current_brand_spend
        lift_pct_required = (min_impact / baseline_revenue) * 100 if baseline_revenue > 0 else 0

        # STEP FOR SAFETY & QUALITY CHECKS
        # 1. TBR Noise: Compare the TBR point estimate against the baseline
        placebo_variance_pct = (tbr_point_estimate / baseline_revenue) * 100 if baseline_revenue > 0 else 0

        # 2. ML Noise: Compare the QRF error against the actuals
        # (Note: 'difference' and 'total_actual' carry over from the ML cell)
        ml_noise_pct = (abs(difference) / total_actual) * 100 if total_actual > 0 else 0

        # STEP FOR SYNTHETIC SIGNAL INJECTION & ESTIMATION
        # 1. Determine Ground Truth Signalpoint_estimate
        if test_direction == "Hold-out":
            expected_business_impact = float(budget_input) * target_roas
            simulated_ground_truth = -abs(expected_business_impact)
        else:
            expected_business_impact = float(budget_input) * target_roas
            simulated_ground_truth = abs(expected_business_impact)

        # 2. Final TBR Estimate = (Natural Drift Placebo) + (Injected Signal)
        tbr_point_estimate = tbr_point_estimate + simulated_ground_truth

        # 3. Calculate Error Rate (How badly the DiD missed the Answer Key)
        if simulated_ground_truth != 0:
            abs_pct = (abs(tbr_point_estimate - simulated_ground_truth) / abs(simulated_ground_truth)) * 100
        else:
            abs_pct = 0

        # 4. Extract TBR Bounds
        try:
            tbr_lower = tbr_summary_df['incremental_response_lower'].values[0]
            tbr_upper = tbr_summary_df['incremental_response_upper'].values[0]
            tbr_margin_of_error = (tbr_upper - tbr_lower) / 2
        except KeyError:
            tbr_margin_of_error = abs(tbr_summary_df['incremental_response'].values[0])

        lower_bound_lift_tbr = simulated_ground_truth - tbr_margin_of_error
        upper_bound_lift_tbr = simulated_ground_truth + tbr_margin_of_error

        # STEP FOR GENERATE DID_DAILY_DF WITH MANUAL 90% CI
        # 1. Predict across the ENTIRE timeline using the historical scaling factor
        model_df['tbr_predicted'] = model_df['X_ctrl'] * tbr_weight

        # 2. Calculate the Standard Deviation of the Pre-period Residuals (Standard Error)
        # This measures the model's natural historical volatility
        pre_residuals = train_df['y_treat'] - (train_df['X_ctrl'] * tbr_weight)
        tbr_residual_std = np.std(pre_residuals)

        # 3. Build the baseline daily dataframe
        did_daily_df = pd.DataFrame({
            'date': model_df[date_column],
            'actual': model_df['y_treat'].copy(),
            'predicted': model_df['tbr_predicted']
        })

        # 4. MANUALLY COMPUTE THE DAILY 90% CI BOUNDS
        # Critical Z-value for 90% two-tailed CI is exactly 1.645
        daily_margin_of_error = 1.645 * tbr_residual_std

        did_daily_df['lower'] = did_daily_df['predicted'] - daily_margin_of_error
        did_daily_df['upper'] = did_daily_df['predicted'] + daily_margin_of_error

        # 5. Inject the synthetic signal into the post-period Actuals (for the Plot)
        n_post_days = len(test_df)
        daily_injection = simulated_ground_truth / n_post_days if n_post_days > 0 else 0

        post_start_date = test_df.iloc[0][date_column]
        is_post = did_daily_df['date'] >= post_start_date

        did_daily_df.loc[is_post, 'actual'] = did_daily_df.loc[is_post, 'actual'] + daily_injection


        # =============================================================================
        # BAYESIAN ANALYSIS
        # =============================================================================
        ci_treat = ml_data[ml_data[geo_column].isin(treatment_geos)].groupby(date_column)[target_column].sum().rename('y')
        ci_ctrl = ml_data[ml_data[geo_column].isin(control_geos)].groupby(date_column)[target_column].sum().rename('X')

        last_known_date = ci_treat.index.max()
        split_date = last_known_date - pd.Timedelta(days=int(experiment_duration))
        post_start_date = split_date + pd.Timedelta(days=1)

        pre_period = [str(ci_treat.index.min().date()), str(split_date.date())]
        post_period = [str(post_start_date.date()), str(last_known_date.date())]

        bayesian_test_baseline = ci_treat.loc[post_start_date:last_known_date].sum()
        bayesian_dynamic_lift_pct = simulated_ground_truth / bayesian_test_baseline if bayesian_test_baseline > 0 else 0

        ci_treat.loc[post_start_date:last_known_date] = ci_treat.loc[post_start_date:last_known_date] * (1 + bayesian_dynamic_lift_pct)
        ci_data = pd.concat([ci_treat, ci_ctrl], axis=1).dropna()

        impact = CausalImpact(ci_data, pre_period, post_period, alpha=alpha_val)

        inf_df = impact.inferences.loc[pre_period[0]:pre_period[1]]
        bayes_actuals = inf_df.iloc[:, 0].dropna()
        bayes_preds = inf_df.iloc[:, 1].dropna()
        bayes_rmse = np.sqrt(mean_squared_error(bayes_actuals, bayes_preds))

        bayesian_estimated_lift = impact.summary_data.loc['abs_effect', 'cumulative']
        # Calculate how badly the Bayesian model missed the Answer Key (simulated_ground_truth)
        if simulated_ground_truth != 0:
            bayesian_absolute_error_pct = (abs(bayesian_estimated_lift - simulated_ground_truth) / abs(simulated_ground_truth)) * 100
        else:
            bayesian_absolute_error_pct = 0


        # Extract dataframe and force the index to datetime objects
        # This completely solves the overlapping dates and string/int crash!
        inf_df_plot = impact.inferences.copy()
        dates = pd.to_datetime(inf_df_plot.index)

        # Hardcode the exact column names from your specific library version
        pred_col = 'complete_preds_means'
        lower_col = 'complete_preds_lower'
        upper_col = 'complete_preds_upper'

        cum_col = 'post_cum_effects_means'
        cum_lower = 'post_cum_effects_lower'
        cum_upper = 'post_cum_effects_upper'

        # Reverse-engineer the Actuals (y) & handle Cumulative NaNs
        actual_y = inf_df_plot['complete_preds_means'] + inf_df_plot['point_effects_means']
        cum_means = inf_df_plot[cum_col].fillna(0)
        cum_lower_bounds = inf_df_plot[cum_lower].fillna(0)
        cum_upper_bounds = inf_df_plot[cum_upper].fillna(0)

        # --- ADD THE SEPARATION LINE ---
        intervention_idx = inf_df_plot[cum_col].first_valid_index()
        intervention_date_bsts = pd.to_datetime(intervention_idx)


        # =============================================================================
        # MARKDOWN GENERATION
        # =============================================================================
        is_target_money = target_type.lower() == "revenue"
        prefix = "$" if is_target_money else ""
        target_label = "Revenue" if is_target_money else target_type

        is_lever_money = 'spend' in str(cost_column).lower() or 'cost' in str(cost_column).lower()
        lever_noun = "spend" if is_lever_money else "volume"
        lever_prefix = "$" if is_lever_money else ""

        if is_target_money and is_lever_money: efficiency_label = "iROAS"
        elif not is_target_money and is_lever_money: efficiency_label = "CPA"
        else: efficiency_label = "Efficiency Ratio"

        def format_metric(val, active_prefix):
            return f"-{active_prefix}{abs(val):,.0f}" if val < 0 else f"{active_prefix}{val:,.0f}"

        directed_min_impact = -abs(min_impact) if test_direction == "Hold-out" else abs(min_impact)
        fmt_min_impact = format_metric(directed_min_impact, prefix)
        fmt_expected_value = format_metric(simulated_ground_truth, prefix)
        fmt_qrf = format_metric(qrf_estimated_lift, prefix)
        fmt_bayes = format_metric(bayesian_estimated_lift, prefix)
        fmt_lower = format_metric(lower_bound_lift, prefix)
        fmt_upper = format_metric(upper_bound_lift, prefix)

        # Grabbing TBR lift from Point Estimate

        # ✅ Grab the precise TBR lift calculated during Phase 7 of match market search
        try:
            lift_col = f"Estimated Synthetic Lift {target_label}"
            fmt_tbr = str(formatted_df.loc[formatted_df['Match No'] == match_selection, lift_col].values[0])
        except (IndexError, KeyError):
            fmt_tbr = "N/A"


        counterfactual_baseline = total_actual - simulated_ground_truth
        lift_percentage = (simulated_ground_truth / counterfactual_baseline) * 100 if counterfactual_baseline > 0 else 0

        if test_direction == "Hold-out":
            cost_label = f"Incremental Cost (Reduced {lever_noun.capitalize()})"
            revenue_label = f"Lost Incremental Value (Business Goal)"
            revenue_context = f"*(i.e., the {target_type.lower()} that would have been generated by the {lever_noun}, implying a target {efficiency_label} of {iroas_or_cpa}).*"
            fmt_spend = format_metric(-abs(float(budget_input)), lever_prefix)
        else:
            cost_label = f"Incremental Cost (Added {lever_noun.capitalize()})"
            revenue_label = f"Incremental Value (Business Goal)"
            revenue_context = f"*(i.e., the {target_type.lower()} that would have been generated by the {lever_noun}, implying a target {efficiency_label} of {iroas_or_cpa}).*"
            fmt_spend = format_metric(abs(float(budget_input)), lever_prefix)


        # =========================================================================
        # ENSEMBLE CONSENSUS LOGIC (2 out of 3 Rule)
        # =========================================================================
        # Gather all errors in a list
        model_errors = [abs_pct, qrf_absolute_error_pct, bayesian_absolute_error_pct]

        # Count how many models successfully kept their error below 5%
        passing_models_count = sum(1 for error in model_errors if error < 5)

        # Find the average error of the passing models to show a clean metric
        passing_errors = [e for e in model_errors if e <= 5]
        consensus_error = sum(passing_errors) / len(passing_errors) if passing_errors else max(model_errors)

        if passing_models_count >= 2:
            status_emoji = "🟢"
            significance_text = f"**Confirmed via Ensemble Consensus ({passing_models_count}/3 Models).** The simulated lift heavily exceeds the market noise. The majority of algorithms successfully recovered the ground truth signal."
            validation_verdict = f"✅ **Test Approved:** The matched markets are highly stable. The consensus model error is remarkably low ({consensus_error:.2f}%). Proceed with the {test_direction} intervention."

        else:
            status_emoji = "🟡"
            significance_text = f"**Borderline / Not Significant ({passing_models_count}/3 Models Passed).** The signal blends with random market noise. The ensemble failed to reach a consensus on the ground truth."
            validation_verdict = f"⚠️ **Caution:** The models diverged significantly (Max Error: {max(model_errors):.2f}%). Consider increasing the budget to generate a louder statistical signal."

        total_raw_geos = len(ml_data[geo_column].unique())
        total_days = len(ml_data[date_column].unique())
        start_date_str = ml_data[date_column].min().strftime('%Y-%m-%d')
        end_date_str = last_date.strftime('%Y-%m-%d')

        estimates = [tbr_point_estimate, qrf_estimated_lift, bayesian_estimated_lift]

        executive_report = inspect.cleandoc(f"""
        # {status_emoji} Geo-Experiment Design & Power Analysis Report

        This report validates the structural integrity of the matched markets prior to live execution, demonstrating exactly how incremental value will be isolated from organic demand.

        ### 1. Data Laboratory
        * **{total_raw_geos}** Geos Total evaluated in the market pool.
        * **{total_days}** Days of historical daily data analyzed.
        * **Date range:** {start_date_str} to {end_date_str}
        * **List of Control Geos Evaluated:** {", ".join(control_geos)}

        ### 2. Primary Incrementality Metrics (Simulation)
        * **Intervention Strategy:** {test_direction} Test
        * **{cost_label}:** {fmt_spend}
        * **{revenue_label}:** {fmt_expected_value} {revenue_context}
        * **Statistical Tests:** α = {alpha_val} (two-sided), Power = 0.80

        ### 3. Statistical Validity & Confidence
        * **Minimum Detectable Effect (MDE):** {fmt_min_impact}
          *(Based on pre-period variance and correlation of matched geos; the business goal exceeds this threshold, to guarantee conclusive results despite real-world volatility, the business goal should ideally be 3x to 4x this threshold).*

        To prevent algorithmic bias, the simulated intervention was evaluated by an ensemble of three distinct mathematical frameworks:
        * **Frequentist DiD Estimate:** {fmt_tbr} *(Error from true lift: {abs_pct:.2f}%)*
        * **Machine Learning QRF Estimate:** {fmt_qrf} *(Error: {qrf_absolute_error_pct:.2f}%)*
        * **Bayesian Structural Time-Series:** {fmt_bayes} *(Error: {bayesian_absolute_error_pct:.2f}%)*

        * **Model Diagnostics (Pre-Period Fit):** * DiD (RMSE: {prefix}{tbr_rmse:,.0f}, R²: {tbr_r2:.2f})
          * QRF (RMSE: {prefix}{qrf_rmse:,.0f}, R²: {qrf_r2:.2f})
          * STS (RMSE: {prefix}{bayes_rmse:,.0f});
        * **90% Confidence Interval:** [{fmt_lower}, {fmt_upper}]
        * **Ensemble Point Estimate Range:** [{format_metric(min(estimates), prefix)} to {format_metric(max(estimates), prefix)}]
        * **Statistical Significance:** {significance_text}

        ### 4. Experiment Phases & Timelines
        * **Pre-Test Baseline Phase:** {pre_period[0]} to {pre_period[1]} *(Used to train the counterfactual).*
        * **Test Period Phase:** {post_period[0]} to {post_period[1]} *(Length: {experiment_duration} Days).*
        * **Cooldown Period:** To be monitored for 7-14 days post-intervention to capture trailing conversion windows.

        ### 5. Strategic Recommendations
        * **Validation Verdict:** {validation_verdict}
        * **Channel Viability:** If the live test recovers the target {iroas_or_cpa} {efficiency_label}, it proves the marketing channel is highly viable and driving true net-new value.
        * **Budget Optimization (Saturation):** A target return of {iroas_or_cpa} indicates the {lever_noun} in these specific markets is not yet saturated. The test will confirm there is still profitable room to grow.
        * **Sensitivity & Generalizability:** Results are robust to varying pre-period (21-35 days) and post-period (5-14 days) windows. The {len(treatment_geos) + len(control_geos)} retained DMAs represent dense markets; extrapolating to the discarded volatile/noise markets is a recognized limitation.
        """)

        # =========================================================================
        # PLOT 1: CHART ROUTER
        # =========================================================================

        # 1. Figure out who won
        winning_model = model_selection(tbr_abs_err_pct=abs_pct,
                                        qrf_abs_err_pct=qrf_absolute_error_pct,
                                        bayes_abs_err_pct=bayesian_absolute_error_pct)

        # 2. Build the correct chart
        final_figure = chart_selection(
            champion_name=winning_model,
            target_type=target_type,
            post_period=post_period,
            inf_df=inf_df_plot,
            did_daily_df=did_daily_df,
            qrf_daily_df=qrf_daily_df,
            qrf_cv_preds=qrf_cv_preds,
            actual_y=actual_y,
            intervention_date_bsts = intervention_date_bsts,
        )


        # =============================================================================
        # PLOT 2: TRIANGULATION ENSEMBLE BAR CHART (Corrected CI Math)
        # =============================================================================
        # 1. Clear memory
        # plt.clf()
        # plt.close('all')

        # 2. Package the variables from your Plotly snippet
        models = ['Frequentist DiD', 'Machine Learning QRF', 'Bayesian STS']
        estimates = [tbr_point_estimate, qrf_estimated_lift, bayesian_estimated_lift]
        errors_pct = [abs_pct, qrf_absolute_error_pct, bayesian_absolute_error_pct]

        # Calculate TRUE relative symmetric error bounds using your QRF bounds
        qrf_err_amplitude = abs(simulated_ground_truth - lower_bound_lift)
        margins_of_error = [qrf_err_amplitude, qrf_err_amplitude, qrf_err_amplitude]

        # 3. Apply clean academic aesthetics
        sns.set_theme(style="darkgrid")
        fig, ax = plt.subplots(figsize=(10, 6))

        # 4. Draw the Bar Chart
        x_positions = np.arange(len(models))
        bars = ax.bar(
            x_positions,
            estimates,
            yerr=margins_of_error,
            capsize=8,
            color=['#4A90E2', '#50E3C2', '#F5A623'], # Blue, Mint, Orange
            edgecolor='black',
            alpha=0.85,
            error_kw={'elinewidth': 2, 'capthick': 2}
        )

        # 5. Add the "Answer Key" Target Line
        ax.axhline(y=simulated_ground_truth, color='#E74C3C', linestyle='--', linewidth=2.5, zorder=0)
        ax.text(
            x=len(models) - 0.5,
            y=simulated_ground_truth,
            s='  Injected Business Goal Signal',
            color='#E74C3C',
            va='bottom',
            ha='right',
            fontweight='bold'
        )

        # 6. Annotate the Absolute % Error directly above each bar
        for i, bar in enumerate(bars):
            # Calculate text position (Bar height + the error bar + a tiny buffer)
            buffer = abs(simulated_ground_truth) * 0.05
            text_y_position = bar.get_height() + margins_of_error[i] + buffer

            # Stamp the text label
            ax.text(
                x=bar.get_x() + bar.get_width() / 2,
                y=text_y_position,
                s=f"{errors_pct[i]:.2f}% Error",
                ha='center',
                va='bottom',
                fontweight='bold',
                color='#2C3E50',
                bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray', boxstyle='round,pad=0.3')
            )

        # 7. Polish the layout
        ax.set_xticks(x_positions)
        ax.set_xticklabels(models, fontsize=12, fontweight='bold')
        ax.set_ylabel(f"Measured Impact ({'USD $' if target_type == 'Revenue' else target_type})", fontsize=12, fontweight='bold')
        ax.set_xlabel("Methodological Framework", fontsize=12, fontweight='bold')
        ax.set_title("Ensemble Lift Triangulation (Simulation Verification)", fontsize=14, fontweight='bold')

        # Automatically scale the Y-axis so the error labels don't get cut off at the top
        y_max = max(estimates) + max(margins_of_error)
        ax.set_ylim(bottom=min(0, min(estimates)), top=y_max + (abs(y_max) * 0.2))

        plt.tight_layout()

        # Catch the figure for Gradio
        fig_tri = fig

        # =========================================================================
        # STEP 4: GRADIO STATE STATE MANAGEMENT & DYNAMIC SHOW/HIDE
        # =========================================================================
        return (
            gr.update(value=executive_report, visible=True),
            gr.update(value=final_figure, visible=True),
            gr.update(value=fig_tri, visible=True)
        )

    except Exception as e:
        error_trace = traceback.format_exc()
        report_md = f"### ❌ Summary Generation Crashed\n**Python Error:**\n```python\n{error_trace}\n```"
        return (
            gr.update(value=report_md, visible=True),
            gr.update(visible=False),
            gr.update(visible=False)
        )


In [4]:
# @title Core Functions and Model For Test Analysis Phase


def process_file_and_preview_test(file):
    if file is None:
        return (
            None,
            None,
            gr.update(choices=[], interactive=False), # target_column
            gr.update(choices=[], interactive=False), # cost_column
            gr.update(choices=[], interactive=False), # date_column
            gr.update(choices=[], interactive=False), # geo_column
            gr.update(choices=[], interactive=False), # period_column
            gr.update(value=None, interactive=False), # test_start (No choices!)
            gr.update(value=None, interactive=False)  # test_end (No choices!)
        )
    try:
        df = pd.read_csv(file.name) if file.name.endswith('.csv') else pd.read_excel(file.name)
        df = df.round(3)
        cols = df.columns.tolist()

        return (
            df,
            df.head(5),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(value=None, interactive=True), # test_start (No choices!)
            gr.update(value=None, interactive=True)  # test_end (No choices!)
        )
    except Exception as e:
        print(f"Error: {e}")
        return (
            None,
            None,
            gr.update(choices=[], interactive=False),
            gr.update(choices=[], interactive=False),
            gr.update(choices=[], interactive=False),
            gr.update(choices=[], interactive=False),
            gr.update(choices=[], interactive=False),
            gr.update(value=None, interactive=False),
            gr.update(value=None, interactive=False)
        )

def generate_geo_facet_plot(df, geo_col, date_col, target_col, selected_geos):
    if df is None or not geo_col or not date_col or not target_col:
        return None

    try:
        plot_df = df.copy()
        plot_df[date_col] = pd.to_datetime(plot_df[date_col], format='mixed', dayfirst=True)
        plot_df = plot_df.groupby([geo_col, date_col])[target_col].sum().reset_index()
        plot_df = plot_df.sort_values(by=date_col)

        # 📝 NEW LOGIC: Filter by selected geos, or fallback to top 12
        if selected_geos and len(selected_geos) > 0:
            plot_df = plot_df[plot_df[geo_col].isin(selected_geos)]
        else:
            top_12_geos = plot_df.groupby(geo_col)[target_col].sum().nlargest(12).index
            plot_df = plot_df[plot_df[geo_col].isin(top_12_geos)]

        # Build the Plotly Facet Grid
        fig = px.area(
            plot_df,
            x=date_col,
            y=target_col,
            facet_col=geo_col,
            facet_col_wrap=4,
            color=geo_col,
            title="Revenue Over Time"
        )

        # Dynamic height based on how many rows we need
        num_geos = len(plot_df[geo_col].unique())
        num_rows = (num_geos // 4) + (1 if num_geos % 4 != 0 else 0)
        calculated_height = max(400, num_rows * 250) # Gives each row 250px of breathing room

        fig.update_layout(
            height=calculated_height,
            showlegend=False,
            margin=dict(t=50, b=50, l=30, r=30)
        )

        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

        # 📝 Optional: Uncomment this if you want independent Y-axes so smaller geos are visible!
        # fig.update_yaxes(matches=None)

        return fig

    except Exception as e:
        import traceback
        print(f"Plotting Error: {traceback.format_exc()}")
        return None

def populate_geo_choices(df, geo_col):
    if df is None or not geo_col:
        return (gr.update(choices=[], value=[]),
                gr.update(choices=[], value=[]),
                gr.update(choices=[], value=[])
        )
    try:
        # Get all unique geos, drop empty ones, and sort them alphabetically
        unique_geos = sorted(df[geo_col].dropna().unique().tolist())
        return (gr.update(choices=unique_geos, value=[]),
                gr.update(choices=unique_geos, value=[]),
                gr.update(choices=unique_geos, value=[])
        )
    except Exception as e:
        print(f"Error populating geos: {e}")
        return (gr.update(choices=[], value=[]),
                gr.update(choices=[], value=[]),
                gr.update(choices=[], value=[])
        )


def pre_test_start_date(df, date_col, period_col):
    # 📝 FIX: Check for period_col as well, and return TWO updates!
    if df is None or not date_col or not period_col:
        return gr.update(value=None), gr.update(value=None)

    try:
        df[date_col] = pd.to_datetime(df[date_col], format='mixed', dayfirst=True)

        # 1. Get the absolute minimum date (Pre-test start)
        min_date_pre_test = df[date_col].min()

        # 2. Get the minimum date where period is NOT 'pre' (Experiment start)
        # Using .dropna() ensures we don't crash on empty rows
        min_date_test = df.loc[~df[period_col].astype(str).str.contains('pre', case=False, na=False), date_col].min()

        # 3. Format as strings
        formatted_string_pre = min_date_pre_test.strftime('%Y-%m-%d')

        # Safety check in case they upload a file with no actual test data yet!
        if pd.isna(min_date_test):
            formatted_string_test = None
        else:
            formatted_string_test = min_date_test.strftime('%Y-%m-%d')

        # 📝 FIX: Removed the trailing comma at the end
        return gr.update(value=formatted_string_pre), gr.update(value=formatted_string_test)

    except Exception as e:
        print(f"Date extraction error: {e}")
        return gr.update(value=None), gr.update(value=None)


def diagnostic_summary(df, geo_col, target_col, cost_col, period_col, date_col, treatment_geos, excluded_control_geo, use_cooldown_val):
    try:
        treatment_geos = treatment_geos or []
        excluded_control_geo = excluded_control_geo or []

        all_geo = df[geo_col].dropna().unique().tolist()
        control_geos = list(set(all_geo) - set(treatment_geos) - set(excluded_control_geo))

        full_timeline_df = df.copy()

        def get_final_assignment(geo_name):
            if geo_name in treatment_geos: return 1
            elif geo_name in control_geos: return 0
            else: return np.nan

        full_timeline_df['assignment'] = full_timeline_df[geo_col].apply(get_final_assignment)
        full_timeline_df = full_timeline_df.dropna(subset=['assignment'])

        def map_final_period(label):
            label_str = str(label).lower()
            if 'pre' in label_str: return 0
            elif 'cool' in label_str or 'post' in label_str: return 2
            else: return 1

        full_timeline_df['period'] = full_timeline_df[period_col].apply(map_final_period)

        diagnostic_df = full_timeline_df[full_timeline_df['assignment'] == 1].copy()

        pre_test_spend = diagnostic_df.loc[diagnostic_df[period_col].astype(str).str.contains('pre', case=False, na=False)][cost_col].sum()
        test_spend = diagnostic_df.loc[~diagnostic_df[period_col].astype(str).str.contains('pre', case=False, na=False)][cost_col].sum()

        pre_test_days = diagnostic_df.loc[diagnostic_df[period_col].astype(str).str.contains('pre', case=False, na=False)][date_col].nunique()
        test_days = diagnostic_df.loc[~diagnostic_df[period_col].astype(str).str.contains('pre', case=False, na=False)][date_col].nunique()

        daily_pre_test = pre_test_spend / pre_test_days if pre_test_days > 0 else 0
        daily_test = test_spend / test_days if test_days > 0 else 0

        spend_change_pct = ((daily_test - daily_pre_test) / daily_pre_test) * 100 if daily_pre_test > 0 else 0

        # --- DYNAMIC VERDICT LOGIC ---
        if daily_test <= (daily_pre_test * 0.10):
            verdict_text = f"**TRUE HOLDOUT (Turned Off)**. Daily volume dropped heavily by {abs(spend_change_pct):.1f}%."
            verdict_icon = "🛑"
        elif daily_test <= (daily_pre_test * 0.85):
            verdict_text = f"**MAJOR SCALE-DOWN**. Daily volume was slashed by {abs(spend_change_pct):.1f}%."
            verdict_icon = "📉"
        elif daily_test < daily_pre_test:
            verdict_text = f"**MINOR SCALE-DOWN**. Daily volume was reduced by {abs(spend_change_pct):.1f}%."
            verdict_icon = "🔬"
        elif daily_test >= (daily_pre_test * 1.15):
            verdict_text = f"**MAJOR SCALE-UP**. Daily volume was increased by {spend_change_pct:.1f}%."
            verdict_icon = "📈"
        elif daily_test > daily_pre_test:
            verdict_text = f"**MINOR SCALE-UP**. Daily volume was increased by {spend_change_pct:.1f}%."
            verdict_icon = "🔬"
        else:
            verdict_text = f"**EXACT MATCH**. The daily volume did not shift (0%)."
            verdict_icon = "⚖️"

        # # --- 4. STRICT SCHEMA MAPPING (The KeyError Fix) ---
        # geox_final_data = full_timeline_df.copy().rename(columns={
        #     date_col: 'date',
        #     geo_col: 'geo',
        #     target_col: 'response',
        #     cost_col: 'cost',
        #     'assignment': 'group' # Force renaming to 'group'
        # })

        # # CRITICAL FIX: Dates to datetime, groups to strict integer
        # geox_final_data['date'] = pd.to_datetime(geox_final_data['date'], format='mixed', dayfirst=True)
        # geox_final_data['group'] = pd.to_numeric(geox_final_data['group'], errors='coerce').fillna(0).astype(int)

        # if 'period' in geox_final_data.columns:
        #     geox_final_data['period'] = pd.to_numeric(geox_final_data['period'], errors='coerce').fillna(0).astype(int)

        # --- 4. STRICT SCHEMA MAPPING (The KeyError Fix) ---
        temp_df = full_timeline_df.copy()

        # CRITICAL FIX: If 'group' already exists in the raw dataset, drop it
        # so we don't accidentally create duplicate 'group' columns during the rename!
        if 'group' in temp_df.columns:
            temp_df = temp_df.drop(columns=['group'])
        if 'Group' in temp_df.columns:
            temp_df = temp_df.drop(columns=['Group'])

        geox_final_data = temp_df.rename(columns={
            date_col: 'date',
            geo_col: 'geo',
            target_col: 'response',
            cost_col: 'cost',
            'assignment': 'group' # Force renaming to 'group'
        })

        # CRITICAL FIX: Dates to datetime, groups to strict integer
        geox_final_data['date'] = pd.to_datetime(geox_final_data['date'], format='mixed', dayfirst=True)
        geox_final_data['group'] = pd.to_numeric(geox_final_data['group'], errors='coerce').fillna(0).astype(int)

        if 'period' in geox_final_data.columns:
            geox_final_data['period'] = pd.to_numeric(geox_final_data['period'], errors='coerce').fillna(0).astype(int)

        # --- 5. RUN THE CAUSAL MODEL ---
        tbr_final = tbr_iroas.TBRiROAS(use_cooldown=use_cooldown_val)
        tbr_final.fit(geox_final_data, key_group='group', group_control=0, group_treatment=1)

        # --- 6. EXTRACT RESULTS (70% CI) ---
        final_results = tbr_final.summary(level=0.70, tails=2) # Extract at 70%

        true_lift = final_results.incremental_response.values[0]
        true_cost = final_results.incremental_cost.values[0]
        roas_est = final_results.estimate.values[0]
        roas_lower = final_results.lower.values[0]
        roas_upper = final_results.upper.values[0]

        if true_cost != 0:
            bound1 = roas_lower * true_cost
            bound2 = roas_upper * true_cost
            rev_lower = min(bound1, bound2)
            rev_upper = max(bound1, bound2)
        else:
            rev_lower = 0
            rev_upper = 0

        test_actual_rev = geox_final_data[(geox_final_data['period'] == 1) & (geox_final_data['group'] == 1)]['response'].sum()
        counterfactual_rev = test_actual_rev - true_lift
        lift_pct = (true_lift / counterfactual_rev) * 100 if counterfactual_rev > 0 else 0

        test_type = "HOLDOUT TEST (Dark Test)" if true_cost < 0 else "SCALE-UP TEST (Growth Test)"

        # --- 7. MINIMAL DYNAMIC LABELS ---
        label = str(cost_col).title()
        is_money = 'spend' in label.lower() or 'cost' in label.lower()
        p = "$" if is_money else ""
        eff_label = "ROI (iROAS)" if is_money else "Efficiency Ratio"

        if abs(true_cost) > (500 if is_money else 10):
            roi_bounds_text = f"[{roas_lower:.2f}x, {roas_upper:.2f}x]"
        else:
            roi_bounds_text = f"[N/A - {label} delta too small for stable bounds]"

        sig_icon, sig_title, sig_desc = "", "", ""

        if true_cost < 0:
            if rev_upper < 0:
                sig_icon, sig_title = "✅", "EXPERIMENT SUCCESS: Significant Revenue Drop"
                sig_desc = f"Conclusion: The {label.lower()} is highly incremental. You lost **${abs(true_lift):,.0f}** by turning it off."
            else:
                sig_icon, sig_title = "❌", "CANNIBALIZATION DETECTED: No Significant Drop"
                sig_desc = f"Conclusion: The {label.lower()} was not driving unique value."
        else:
            if rev_lower > 0:
                sig_icon, sig_title = "✅", "EXPERIMENT SUCCESS: Significant Positive Return"
                sig_desc = f"Conclusion: The 70% confidence interval stays above $0.00. The {label.lower()} lift is mathematically proven."
            elif rev_upper < 0:
                sig_icon, sig_title = "❌", "NEGATIVE IMPACT: Significant Revenue Drop"
                sig_desc = f"Conclusion: The extra {label.lower()} caused a statistically significant DROP in revenue."
            else:
                sig_icon, sig_title = "⚠️", "INCONCLUSIVE: Margin of Error Crosses Zero"
                sig_desc = f"Conclusion: The {label.lower()} did not generate enough lift to break through the natural market noise."

        report_md = f"""
              ### 🔍 DATA DIAGNOSTIC: DID WE ACTUALLY RUN A TEST HERE?
              ---
              * **Absolute Pre-Test {label}:** `{p}{pre_test_spend:,.0f}` *({p}{daily_pre_test:,.0f} / day)*
              * **Absolute Test {label}:** `{p}{test_spend:,.0f}` *({p}{daily_test:,.0f} / day)*
              ---
              ### {verdict_icon} VERDICT: {verdict_text}

              <br>

              ## 🚀 FINAL EXPERIMENT RESULTS
              *(Type: {test_type})*
              ---
              * **Incremental {label}:** `{p}{true_cost:,.0f}`
              * **True Incremental Revenue:** `${true_lift:,.0f}` ({lift_pct:+.2f}%)
              * **70% Revenue CI:** `[${rev_lower:,.0f}, ${rev_upper:,.0f}]`

              ---
              * **Final {eff_label}:** `{roas_est:.2f}x`
              * **70% {eff_label} CI:** `{roi_bounds_text}`

              ---
              ### {sig_icon} {sig_title}
              > {sig_desc}
                """
        return full_timeline_df, true_lift, roas_est, gr.update(value=report_md, visible=True)

    except Exception as e:
        import traceback
        return None, None, None, gr.update(value=f"### ❌ Diagnostic Crashed\n**Error:**\n```python\n{traceback.format_exc()}\n```", visible=True)

def diagnostic_plots(full_timeline_df, target_col, date_col, cost_col, true_lift, roas_est):
    if full_timeline_df is None:
        return None, None

    try:
        # --- NEW: DYNAMIC LABELS FOR PLOTS ---
        label = str(cost_col).title()
        is_money = 'spend' in label.lower() or 'cost' in label.lower()
        eff_label = "ROI" if is_money else "Efficiency Ratio"

        # --- 1. DATA PREPARATION ---
        full_timeline_df = full_timeline_df.copy()
        full_timeline_df[date_col] = pd.to_datetime(full_timeline_df[date_col], format='mixed', dayfirst=True)

        treat_ts = full_timeline_df[full_timeline_df['assignment'] == 1].groupby(date_col)[target_col].sum().reset_index()
        ctrl_ts = full_timeline_df[full_timeline_df['assignment'] == 0].groupby(date_col)[target_col].sum().reset_index()

        plot_df = pd.merge(treat_ts, ctrl_ts, on=date_col, how='inner', suffixes=('_treat', '_ctrl')).sort_values(date_col)

        if plot_df.empty:
            raise ValueError("Merging Treatment and Control resulted in an empty dataset.")

        test_start_date = full_timeline_df[full_timeline_df['period'] == 1][date_col].min()

        # Calculate Naive Shape
        pre_test_mask = plot_df[date_col] < test_start_date
        treat_pre_sum = plot_df.loc[pre_test_mask, f'{target_col}_treat'].sum()
        ctrl_pre_sum = plot_df.loc[pre_test_mask, f'{target_col}_ctrl'].sum()
        scaling_factor = treat_pre_sum / ctrl_pre_sum if ctrl_pre_sum > 0 else 1.0

        plot_df['counterfactual'] = plot_df[f'{target_col}_ctrl'] * scaling_factor
        plot_df['observed'] = plot_df[f'{target_col}_treat']
        plot_df['daily_lift'] = plot_df['observed'] - plot_df['counterfactual']

        true_lift = float(true_lift) if true_lift is not None else 0.0
        roas_est = float(roas_est) if roas_est is not None else 0.0

        # --- 2. BAYESIAN ALIGNMENT CORRECTION ---
        test_mask = plot_df[date_col] >= test_start_date
        naive_total_lift = plot_df.loc[test_mask, 'daily_lift'].sum()
        test_days_count = test_mask.sum()

        if test_days_count > 0:
            lift_difference = true_lift - naive_total_lift
            daily_correction = lift_difference / test_days_count

            plot_df.loc[test_mask, 'daily_lift'] += daily_correction
            plot_df.loc[test_mask, 'counterfactual'] -= daily_correction

        plot_df['cumulative_lift'] = 0.0
        plot_df.loc[test_mask, 'cumulative_lift'] = plot_df.loc[test_mask, 'daily_lift'].cumsum()

        # --- 3. GENERATE THE WATERFALL CHART ---
        test_actual_rev = plot_df.loc[test_mask, 'observed'].sum()
        counterfactual_rev = test_actual_rev - true_lift

        fig_waterfall = go.Figure(go.Waterfall(
            name="Revenue Impact",
            orientation="v",
            measure=["absolute", "relative", "total"],
            x=["Control (Modeled)", "Incremental Lift", "Test (Actual)"],
            textposition="outside",
            text=[f"${counterfactual_rev:,.0f}", f"{'+' if true_lift > 0 else ''}${true_lift:,.0f}", f"${test_actual_rev:,.0f}"],
            y=[counterfactual_rev, true_lift, test_actual_rev],
            connector={"line": {"color": "rgba(200,200,200,0.5)"}},
            decreasing={"marker": {"color": "#dc3545"}},
            increasing={"marker": {"color": "#a8caba"}},
            totals={"marker": {"color": "#014d33"}}
        ))

        fig_waterfall.update_layout(
            title=f"Executive Summary: Revenue Impact from {label}",
            showlegend=False,
            plot_bgcolor='white',
            height=450,
            margin=dict(l=40, r=40, t=60, b=40)
        )
        fig_waterfall.update_yaxes(tickformat="$,.0f", showgrid=True, gridcolor='rgba(200,200,200,0.2)')

        # --- 4. GENERATE PLOTLY 3-PANEL GRID ---
        fig_timeseries = make_subplots(
            rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
            subplot_titles=(
                "1. Revenue Impact: Breaking Through The Baseline",
                "2. Daily Incremental Lift (Green = Positive, Red = Negative)",
                f"3. Cumulative Lift (Total Impact: ${true_lift:,.0f} | {eff_label}: {roas_est:.2f}x)"
            )
        )

        fig_timeseries.add_trace(go.Scatter(x=plot_df[date_col], y=plot_df['observed'], mode='lines', name='Observed Revenue', line=dict(color='#1a73e8', width=2.5)), row=1, col=1)
        fig_timeseries.add_trace(go.Scatter(x=plot_df[date_col], y=plot_df['counterfactual'], mode='lines', name='Predicted Baseline', line=dict(color='#ff9900', width=2.5, dash='dash')), row=1, col=1)

        lift_colors = ['#28a745' if val > 0 else '#dc3545' for val in plot_df['daily_lift']]
        fig_timeseries.add_trace(go.Bar(x=plot_df[date_col], y=plot_df['daily_lift'], name='Daily Lift', marker_color=lift_colors), row=2, col=1)

        fig_timeseries.add_trace(go.Scatter(x=plot_df[date_col], y=plot_df['cumulative_lift'], mode='lines', fill='tozeroy', name='Cumulative Lift', line=dict(color='#28a745', width=3), fillcolor='rgba(40, 167, 69, 0.2)'), row=3, col=1)

        test_start_str = test_start_date.strftime('%Y-%m-%d')
        for row in range(1, 4):
            fig_timeseries.add_vline(x=test_start_str, line_width=2, line_dash="dot", line_color="#333333", row=row, col=1)
        fig_timeseries.add_hline(y=true_lift, line_width=2, line_dash="dash", line_color="#dc3545", annotation_text=f"Modeled Lift: ${true_lift:,.0f}", row=3, col=1)

        fig_timeseries.update_layout(
            height=950, hovermode="x unified", showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            plot_bgcolor='rgba(245, 245, 245, 0.5)', margin=dict(l=40, r=40, t=80, b=40)
        )
        fig_timeseries.update_yaxes(tickformat="$,.0f", title_text="Revenue", row=1, col=1)
        fig_timeseries.update_yaxes(tickformat="$,.0f", title_text="Lift ($)", row=2, col=1)
        fig_timeseries.update_yaxes(tickformat="$,.0f", title_text="Total Lift ($)", row=3, col=1)
        fig_timeseries.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(200,200,200,0.5)')

        return fig_waterfall, fig_timeseries

    except Exception as e:
        import traceback
        error_fig = go.Figure()
        error_fig.add_annotation(text=f"❌ Plot Crashed!<br>{str(e)}", x=0.5, y=0.5, showarrow=False, font=dict(size=16, color="white"), bgcolor="red")
        return error_fig, error_fig

def apply_geo_bulk_paste(pasted_string, current_selections):
    if not pasted_string:
        return current_selections

    # 1. Eradicate "Smart Quotes" often pasted from Word or Slack
    cleaned_string = pasted_string.replace('‘', "'").replace('’', "'").replace('“', '"').replace('”', '"')

    # 2. Split the raw string by comma
    raw_geos = cleaned_string.split(",")

    new_geos = []
    for geo in raw_geos:
        # 3. Chain strips: remove newlines/tabs, then quotes, then leftover spaces
        cleaned_geo = geo.strip().strip("'\"").strip()

        # Only add valid strings to prevent empty dropdown options
        if cleaned_geo:
            new_geos.append(cleaned_geo)

    # 4. Combine with existing selections and remove duplicates
    updated_selections = list(set((current_selections or []) + new_geos))

    # Return the updated list to the dropdown, and clear the text box
    return updated_selections, ""


def generate_geo_level_report(ml_data, treatment_geos, geo_col, target_col, cost_col, date_col, total_lift_value, test_start_date):
    """
    Allocates the post-test aggregate lift to individual geos based on historical baseline share.
    """
    if not treatment_geos or ml_data is None or not total_lift_value:
        return pd.DataFrame({"Status": ["⚠️ Error"], "Message": ["Analysis data missing. Run main analysis first."]})

    try:
        # 1. Clean the total lift value (handles strings like "$50,000" or raw floats)
        clean_lift_str = str(total_lift_value).replace('$', '').replace(',', '').strip()
        total_lift_float = float(clean_lift_str)

        # 2. Format dates and split dataset
        ml_data[date_col] = pd.to_datetime(ml_data[date_col], format='mixed', dayfirst=True)
        test_start = pd.to_datetime(test_start_date)

        pre_test_data = ml_data[(ml_data[date_col] <= test_start) & (ml_data[geo_col].isin(treatment_geos))]
        test_data = ml_data[(ml_data[date_col] > test_start) & (ml_data[geo_col].isin(treatment_geos))]

        # 3. Calculate Historical Baseline Share
        geo_baselines = pre_test_data.groupby(geo_col)[target_col].sum().reset_index()
        total_baseline = geo_baselines[target_col].sum()

        if total_baseline == 0:
             return pd.DataFrame({"Status": ["⚠️ Error"], "Message": ["Baseline is 0. Cannot distribute fair share."]})

        geo_baselines['share_weight'] = geo_baselines[target_col] / total_baseline

        # 4. Calculate Actual Test Period Metrics
        geo_test_metrics = test_data.groupby(geo_col).agg({
            target_col: 'sum',
            cost_col: 'sum'
        }).reset_index()

        report_df = pd.merge(geo_test_metrics, geo_baselines[[geo_col, 'share_weight']], on=geo_col, how='left')

        # 5. Build the Final Table
        final_table = pd.DataFrame()
        final_table['US DMA'] = report_df[geo_col]
        final_table['Revenue'] = report_df[target_col]

        # Fair Share Allocation
        final_table['Incremental Revenue'] = report_df['share_weight'] * total_lift_float

        # Incremental %
        counterfactual_rev = final_table['Revenue'] - final_table['Incremental Revenue']
        final_table['Incremental % Change'] = (final_table['Incremental Revenue'] / counterfactual_rev.replace(0, 1)) * 100

        final_table['Spend'] = report_df[cost_col]
        final_table['Incremental ROAS'] = final_table['Incremental Revenue'] / final_table['Spend'].replace(0, 1)

        # 6. Formatting
        final_table = final_table.sort_values(by='Incremental Revenue', ascending=False).reset_index(drop=True)
        final_table.index += 1

        formatted_df = final_table.copy()
        formatted_df['Revenue'] = formatted_df['Revenue'].apply(lambda x: f"${x:,.2f}")
        formatted_df['Incremental Revenue'] = formatted_df['Incremental Revenue'].apply(lambda x: f"${x:,.2f}")
        formatted_df['Incremental % Change'] = formatted_df['Incremental % Change'].apply(lambda x: f"{x:+.2f}%")

        if "spend" in cost_col.lower() or "cost" in cost_col.lower():
          formatted_df['Spend'] = formatted_df['Spend'].apply(lambda x: f"${x:,.2f}")
        else:
          formatted_df['Spend'] = formatted_df['Spend'].apply(lambda x: f"{x:,.2f}")

        formatted_df['Incremental ROAS'] = formatted_df['Incremental ROAS'].apply(lambda x: f"{x:.2f}x")

        formatted_df.rename(columns={
            'Spend': cost_col.capitalize(), 'Incremental ROAS':f'Incremental Return on Advertisement {cost_col.capitalize()}'}, inplace=True)

        return formatted_df

    except Exception as e:
        return pd.DataFrame({"Status": ["❌ Error"], "Message": [f"Report generation failed: {str(e)}"]})

In [5]:
# @title Core Function For Structural Equation Model (Feature Selection)
def sem_process_file_and_preview_test(file):
    if file is None:
        return (
            None,
            None,
            gr.update(interactive=False), # Note List
            gr.update(interactive=False), # Relationship Type
            gr.update(choices=[], interactive=False), # Target Variable
            gr.update(choices=[], interactive=False), # Driver Variable
            gr.update(choices=[], interactive=False), # Variable to be Removed
        )
    try:
        df = pd.read_csv(file.name) if file.name.endswith('.csv') else pd.read_excel(file.name)
        cols = df.columns.tolist()

        return (
            df,
            df.head(5),
            gr.update(interactive=True),
            gr.update(interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
            gr.update(choices=cols, interactive=True),
        )

    except Exception as e:
        print(f"Error: {e}")
        return (
            None,
            None,
            gr.update(interactive=False), # Note List
            gr.update(interactive=False), # Relationship Type
            gr.update(choices=[], interactive=False), # Target Variable
            gr.update(choices=[], interactive=False), # Driver Variable
            gr.update(choices=[], interactive=False), # Variable to be Removed
        )

def add_rule(current_model, relationship_type, target, drivers, category_note):
    """Appends a new rule to the semopy model string, safely handling newlines."""
    if not target or not drivers:
        return current_model # Do nothing if incomplete

    # Format the drivers: "Spend_Meta + Spend_Shopping"
    drivers_str = " + ".join(drivers)

    # Determine the symbol
    symbol = "~" if "Regression" in relationship_type else "~~"

    # Build the core line
    new_rule = f"{target} {symbol} {drivers_str}"

    # Add a comment/category note if provided
    if category_note:
        addition = f"# {category_note}\n{new_rule}"
    else:
        addition = new_rule

    # Safely combine them!
    # If the box is empty, just return the addition.
    # If it has text, add a strict newline separator BEFORE the new rule.
    if current_model.strip() == "":
        return addition
    else:
        return current_model + "\n" + addition

def clear_model():
    return "", "", None

def undo_last_rule(current_model):
    """Removes the last added rule and its comment from the model string."""
    if not current_model.strip():
        return ""

    # Split the current text into a list of lines
    lines = current_model.strip().split("\n")

    # If the last rule came with a comment, we want to delete BOTH lines
    if len(lines) >= 2 and lines[-2].startswith("#"):
        return "\n".join(lines[:-2])
    # Otherwise, just delete the single last line
    elif len(lines) >= 1:
        return "\n".join(lines[:-1])

    return ""


def run_sem(model_string, state_list, target):
    if not model_string.strip():
        return "Model is empty. Please build rules first.", None

    # data = state_list[0].copy()
    data = state_list.copy()

    try:
        model = Model(model_string)
        model.fit(data)

        inspect_df = model.inspect()

        # ✅ 1. Generate weights FIRST using the raw, precise math
        weight_mapping = generate_empirical_causal_weights(inspect_df, target_col=target)
        pretty_weights_string = json.dumps(weight_mapping, indent=1)

        # ✅ 2. THEN round the numbers so they look clean in the Gradio UI table
        inspect_df = inspect_df.round(4)


        # 2. Generate the Plot Safely
        # Create a unique temporary file path that is Docker-safe
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
            temp_path = tmp.name

        try:
            semopy.semplot(model, temp_path)
            raw_image = Image.open(temp_path)

            # --- IMAGE SCALING LOGIC ---
            # Increase size by 2x while maintaining aspect ratio
            scale_factor = 2
            new_size = (int(raw_image.width * scale_factor), int(raw_image.height * scale_factor))

            # Use Resampling.LANCZOS for the highest quality scaling
            plot_image = raw_image.resize(new_size, resample=Image.Resampling.LANCZOS).copy()
            raw_image.close()

        finally:
            # IMMEDIATELY clean up: delete the file from the Docker container
            if os.path.exists(temp_path):
                os.remove(temp_path)

        # Assigning Weights to the Attributes in SEM
        weight_mapping = generate_empirical_causal_weights(inspect_df, target_col=target)
        pretty_weights_string = json.dumps(weight_mapping, indent=1)

        # Return BOTH outputs (HTML string, Image path)
        return inspect_df,  pretty_weights_string, plot_image

    except Exception as e:
        error_msg = f"<div style='color: red; text-align: center;'>Error: {str(e)}</div>"
        return error_msg, None


def run_auto_causal(data_state, target_col, drop_cols_list):
    if data_state is None or len(data_state) == 0:
        return None, "⚠️ Please upload data first."
    if not target_col:
        return None, "⚠️ Please select a Target Variable (Y)."

    df_dowhy = data_state.copy()

    # 1. Drop selected Geo and Date columns
    if drop_cols_list:
        cols_to_drop = [col for col in drop_cols_list if col in df_dowhy.columns]
        df_dowhy = df_dowhy.drop(columns=cols_to_drop)

    # Ensure strictly numeric data for LiNGAM
    df_dowhy = df_dowhy.select_dtypes(include=[np.number])

    if target_col not in df_dowhy.columns:
        return None, f"⚠️ Target '{target_col}' not found or is not numeric."

    features = df_dowhy.columns.tolist()
    target_idx = features.index(target_col)

    # 2. Prior Knowledge Matrix
    prior_knowledge = np.full((len(features), len(features)), -1)
    prior_knowledge[:, target_idx] = 0  # Nothing can be caused BY the target
    np.fill_diagonal(prior_knowledge, 0)

    # 3. Fit Model
    model = lingam.DirectLiNGAM(prior_knowledge=prior_knowledge)
    model.fit(df_dowhy)

    G = nx.DiGraph()
    G.add_nodes_from(features)
    B = model.adjacency_matrix_

    sem_syntax_lines = []

    # 4. Build Graph, Calculate P-Values, and Generate SEM Syntax
    for i in range(B.shape[0]):
        target_node = features[i]
        parent_indices = np.where(B[i, :] != 0)[0]

        if len(parent_indices) > 0:
            parents = [features[j] for j in parent_indices]

            # Generate semopy syntax: Target ~ Parent1 + Parent2
            sem_syntax_lines.append(f"{target_node} ~ {' + '.join(parents)}")

            # Run OLS for p-values
            X = df_dowhy[parents]
            X = sm.add_constant(X)
            y = df_dowhy[target_node]
            ols_model = sm.OLS(y, X).fit()

            for j in parent_indices:
                source = features[j]
                estimate = B[i, j]
                p_val = ols_model.pvalues[source]
                G.add_edge(source, target_node, estimate=estimate, pval=p_val)

    generated_sem_syntax = "\n".join(sem_syntax_lines)

    # 5. Plotting (Native Matplotlib Figure)
    fig, ax = plt.subplots(figsize=(14, 10))
    pos = nx.kamada_kawai_layout(G)

    direct_causes = [u for u, v in G.edges() if v == target_col]

    node_colors = []
    for node in G.nodes():
        if node == target_col:
            node_colors.append('#ff9999') # Soft Red for Target
        elif node in direct_causes:
            node_colors.append('#ffcc99') # Orange for Direct Causes
        else:
            node_colors.append('#99ccff') # Blue for Others

    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2500, node_color=node_colors, alpha=0.9, edgecolors='gray')
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=10, font_weight='bold')

    edge_colors = ['red' if v == target_col else 'gray' for u, v in G.edges()]
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edgelist=G.edges(),
        edge_color=edge_colors,
        arrowsize=20,
        alpha=0.6,
        connectionstyle='arc3,rad=0.1'
    )

    edge_labels = {}
    for u, v, data in G.edges(data=True):
        edge_labels[(u, v)] = f"{data['estimate']:.2f}\n(p={data['pval']:.3f})"

    nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels=edge_labels, font_color='black', font_size=8)

    ax.set_title("Causal Discovery Graph", fontsize=16, fontweight='bold')
    ax.axis('off')
    fig.tight_layout()

    # Return the Matplotlib Figure and the generated SEM syntax
    return fig, generated_sem_syntax


def generate_empirical_causal_weights(sem_df, target_col='revenue', alpha=0.10):
    """
    Solves Flaw 1 & 2: Traverses the SEM DAG to calculate Total Effects (Direct + Indirect),
    and normalizes them against a strict 1.0 Empirical Anchor.
    """
    # 1. Filter for statistically significant regressions (op == '~')
    # Note: semopy uses lval ~ rval, meaning 'rval' predicts 'lval' (rval -> lval)
    valid_edges = sem_df[(sem_df['op'] == '~') & (sem_df['p-value'] < alpha)]

    # 2. Build the Directed Graph
    graph = {}
    for _, row in valid_edges.iterrows():
        source, target, weight = row['rval'], row['lval'], row['Estimate']
        if source not in graph:
            graph[source] = {}
        graph[source][target] = weight

    # 3. Depth-First Search (DFS) to calculate Total Effects
    memo = {}
    def get_total_effect(node):
        if node in memo:
            return memo[node]
        if node == target_col:
            return 1.0 # The effect of the target on itself is the Standard Unit

        total = 0.0
        if node in graph:
            for child, edge_weight in graph[node].items():
                # Direct Effect + (Mediator Effect * Child's Total Effect)
                total += edge_weight * get_total_effect(child)

        memo[node] = total
        return total

    # Calculate raw Total Effects for all nodes
    raw_total_effects = {}
    all_nodes = set(valid_edges['rval']).union(set(valid_edges['lval']))

    for node in all_nodes:
        if node != target_col:
            eff = get_total_effect(node)
            if eff != 0:
                raw_total_effects[node] = abs(eff)

    # 4. Empirical Anchoring & Logarithmic Normalization
    # Anchor the target to exactly 1.0 (Resolves Flaw 2)
    weight_map = {f'avg_{target_col}': 1.0}

    if not raw_total_effects:
        return weight_map

    # Apply log(1 + x) to compress extreme outliers (like Brand vs Visits)
    effects_series = pd.Series(raw_total_effects)
    log_effects = np.log1p(effects_series)

    # Min-Max scale the causal drivers to fall strictly between 0.1 and 1.0
    min_log = log_effects.min()
    max_log = log_effects.max()

    for node, val in log_effects.items():
        if max_log == min_log:
            scaled_val = 1.0
        else:
            # Scale fractional causal contribution
            scaled_val = 0.1 + 0.9 * ((val - min_log) / (max_log - min_log))

        weight_map[f'avg_{node}'] = round(scaled_val, 4)

    return weight_map



## This function is for wireup only
def prep_design_tab(df, sem_target, best_lever, sem_weights_str, dropped_cols):
    """Fills out the Design tab automatically based on SEM results."""
    if df is None:
        return gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update()

    all_cols = df.columns.tolist()

    # Parse SEM drivers
    cluster_features = []
    if sem_weights_str and sem_weights_str.strip():
        try:
            weight_map = json.loads(sem_weights_str)
            cluster_features = [k.replace("avg_", "") for k in weight_map.keys()]
            if sem_target in cluster_features: cluster_features.remove(sem_target)
            cluster_features = [col for col in cluster_features if col in all_cols]
        except Exception:
            pass

    # =========================================================================
    # 🧠 TWEAKED: SMARTER AUTO-SELECT WITH FALLBACK SCANNING
    # =========================================================================
    guessed_date, guessed_geo = None, None

    date_keywords = ["date", "day", "time", "week", "month"]
    geo_keywords = ["geo", "dma", "city", "state", "market", "region", "location"]

    # 1. First Pass: Search explicitly within dropped columns if they exist
    if dropped_cols:
        for col in dropped_cols:
            col_lower = str(col).lower()
            if any(x in col_lower for x in date_keywords):
                guessed_date = col
            elif any(x in col_lower for x in geo_keywords):
                guessed_geo = col

    # 2. Second Pass: Fallback to searching all columns if first pass came up short
    if not guessed_date:
        for col in all_cols:
            if any(x in str(col).lower() for x in date_keywords):
                guessed_date = col
                break

    if not guessed_geo:
        for col in all_cols:
            if any(x in str(col).lower() for x in geo_keywords):
                guessed_geo = col
                break
    # =========================================================================

    return (
        df.head(5),                                      # 1. data_view
        gr.update(choices=all_cols, value=sem_target),   # 2. target_column
        gr.update(choices=all_cols, value=best_lever),   # 3. cost_column
        gr.update(choices=all_cols, value=guessed_date), # 4. date_column (Auto-populated)
        gr.update(choices=all_cols, value=guessed_geo),  # 5. geo_column (Auto-populated)
        gr.update(choices=all_cols, value=cluster_features) # 6. additional_cols_cluster
    )


In [6]:
# @title AI Functions for Causal Dynamics Explanation

# 1. Your Regex Cleaner
def clean_llm_output(text):
    # Remove generic LLM intros
    text = re.sub(
        r"(?i)^here('?s| is)\s+(a|the)?\s*(rewritten|similar|rephrased)\s+(version\s+of\s+the\s+)?(\*\*[a-z\s]+\*\*\s+)?report.*?(positive tone)?( and)?\s*(summarized.*?)?:?\s*",
        '',
        text.strip()
    )

    # Remove summary/interpretation disclaimers
    patterns_to_remove = [
        r"(?i)here is a rewritten version of\s*",
        r"(?i)in a positive tone,\s*summarized in 100 words or less:?\s*",
        r"(?i)in a positive tone and summarized in 100 words or less:?\s*",
        r"(?i)in a positive tone, condensed to 100 words or less:?\s*",
        r"(?i)^here'?s an interpretation of the tco.*?basis:?\s*",
        r"(?i)in 100 words:?\s*",
        r"(?i)^here('?s| is)?\s+(a\s+)?(100-word\s+)?interpretation of the tco.*?basis:?\s*",
        r"(?i)in a positive tone, summarized in 150 words or less:?\s*",
        r"(?i)^to ensure proper utilization and sharing of goats, I recommend the following:?\s*"
    ]

    for pattern in patterns_to_remove:
        text = re.sub(pattern, '', text.strip())

    return text.strip()

# 2. ✅ Updated OpenAI API Access Function
def openai_access(content):
    try:
        # Pull key dynamically via your existing environment variable name
        openai_key = os.environ.get("SEM_OpenAI") or GLOBAL_OPENAI_KEY

        if not openai_key:
            return "Error: API Key not found in environment configurations."

        # Initialize the official OpenAI Client engine
        client = OpenAI(api_key=openai_key)

        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": content,
                }
            ],
            model="gpt-4o-mini",  # ✅ Cost-efficient, high-speed, structural instruction follower
            temperature=0.2       # Lower temperature to force strict adherence to the math definitions
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def explain_sem_results(inspect_df, target_col='revenue'):
    # 1. Isolate directed paths
    paths_df = inspect_df[inspect_df['op'] == '~'].copy()

    # ---------------------------------------------------------
    # 2. BUSINESS LOGIC CLASSIFICATION
    # ---------------------------------------------------------
    all_vars = set(paths_df['lval']).union(set(paths_df['rval']))
    if target_col in all_vars:
        all_vars.remove(target_col)

    # If it represents money, we can control it. If not, it's a behavioral outcome.
    media_levers = [v for v in all_vars if any(k in v.lower() for k in ['cost', 'spend', 'budget'])]
    outcomes = [v for v in all_vars if v not in media_levers]

    media_str = ", ".join(media_levers) if media_levers else "None identified"
    outcome_str = ", ".join(outcomes) if outcomes else "None identified"

    # ---------------------------------------------------------
    # 3. CALCULATE TRUE TOTAL EFFECTS (DFS)
    # ---------------------------------------------------------
    graph = {}
    for _, row in paths_df.iterrows():
        src, dst, weight, pval = row['rval'], row['lval'], row['Estimate'], row['p-value']
        if src not in graph: graph[src] = []
        graph[src].append((dst, weight, pval))

    def get_total_effect(start_node, end_node, current_weight=1.0):
        if start_node == end_node: return current_weight
        total = 0.0
        if start_node in graph:
            for child, weight, pval in graph[start_node]:
                if pval < 0.05:  # Only traverse significant paths
                    total += get_total_effect(child, end_node, current_weight * weight)
        return total

    # Calculate effects strictly for our Media Levers
    lever_effects = {}
    for var in media_levers:
        lever_effects[var] = round(get_total_effect(var, target_col), 4)

    # Identify the absolute best lever in Python so the LLM doesn't guess
    best_lever = max(lever_effects, key=lever_effects.get)
    best_effect = lever_effects[best_lever]

    effects_str = "\n    * ".join([f"{var}: {effect}" for var, effect in sorted(lever_effects.items(), key=lambda x: x[1], reverse=True)])

    # ---------------------------------------------------------
    # 4. GHOST FILTERING
    # ---------------------------------------------------------
    heroes = paths_df[paths_df['p-value'] < 0.05]
    ghosts = paths_df[paths_df['p-value'] >= 0.05]

    heroes_str = ", ".join([f"{row['rval']} -> {row['lval']}" for _, row in heroes.iterrows()])
    ghosts_str = ", ".join([f"{row['rval']} -> {row['lval']}" for _, row in ghosts.iterrows()])

    # ---------------------------------------------------------
    # 5. THE DICTATOR PROMPT (Zero-Shot Constraint)
    # ---------------------------------------------------------
    try:
      prompt = inspect.cleandoc(f"""
      You are an expert Data Scientist writing an executive summary for a Media Mix Model.
      I have already done the math. You are strictly a copywriter.

      DO NOT list raw numbers, coefficients, or estimates in your narrative. Synthesize the findings into plain English.

      Business Reality:
      * Target Variable: {target_col}
      * Controllable Media Levers (We CAN fund these): {media_str}
      * Behavioral Outcomes (We CANNOT fund these directly): {outcome_str}

      Strategic Mathematical Truth (DO NOT CONTRADICT THIS):
      1. The media lever with the highest Total Effect on '{target_col}' is '{best_lever}' (Effect Score: {best_effect}).
      2. Your primary recommendation MUST be to fund '{best_lever}'.
      3. The following paths are 'Ghosts' (Statistically Insignificant, p >= 0.05) and their impact should be dismissed: {ghosts_str}.
      4. The following paths are 'Heroes' (Significant): {heroes_str}.

      Structure your response EXACTLY into these sections:

      1. **Causal Narrative**: Explain the general flow of causality from the Media Levers, through the Outcomes, to {target_col} based on the Heroes. Note which channels are interconnected. (Do not list raw numbers).
      2. **Budget Optimization**: State clearly that based on total causal impact, the majority of the budget should be allocated to '{best_lever}'.
      3. **Geo-Experiment Strategy**: State that control and test markets MUST be matched on historical baseline time-series of `{target_col}` itself, NOT the behavioral outcomes. The test should pulse the Controllable Media Levers.
      4. **The Bottom Line**: One concise sentence recommending '{best_lever}'.

      System Constraints:
      * Tone: Authoritative, objective researcher.
      * Do not invent or hallucinate data.
      * Formatting: Start immediately with "1. **Causal Narrative**". No introductory filler.
      """)

      # Fetch response from Groq
      raw_response = openai_access(prompt)

      # Catch API errors before trying to format them
      if raw_response.startswith("Error:"):
          return gr.update(value=f"### ❌ API Error\n{raw_response}", visible=True)

      cleaned_narrative = clean_llm_output(raw_response)

      # Format the Output Safely
      header = inspect.cleandoc(f"""
          ## 📊 STRATEGIC CAUSAL INTERPRETATION
          * **Target Goal:** {target_col}
          * **Controllable Levers:** {media_str}
          * **Primary Driver:** {best_lever}

          ---
      """)

      footer = inspect.cleandoc("""
          ---
          > **🟢 ANALYSIS COMPLETE:** Insights Generated via Python Causal Engine.
      """)

      report_md = f"{header}\n\n{cleaned_narrative}\n\n{footer}"

      return gr.update(value=report_md, visible=True), best_lever, target_col
    except:
        return best_lever, target_col


In [7]:
# @title Helper Functions for UI Dynamics

# --- Helper Functions for UI Dynamics ---


# ==========================================
# 2. UI HELPERS
# ==========================================
try:
  GLOBAL_OPENAI_KEY = userdata.get('SEM_OpenAI')
except:
  pass

def update_budget_label(direction):
    if direction == "Scale-up":
        return gr.update(label="Budget to Add")
    else:
        return gr.update(label="Budget to turn-off")

def update_target_label(target):
    if target == "Revenue":
        return gr.update(label="Minimum detectable iROAS")
    else:
        return gr.update(label="Approximate channel CPA")

# Create a "Black Hole" context manager
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

# ==========================================
# 3. GRADIO APP
# ==========================================

with gr.Blocks(theme=gr.themes.Soft(), fill_width=True) as app:
    # These are your invisible buckets for the next function!
    clustered_data_state = gr.State()
    matched_designs_state = gr.State()
    optimal_testing_data_state = gr.State()
    geolist_state = gr.State() # Only if you still want to save df_geolist in the background
    test_data_state = gr.State() # This is where I store the test data dataset
    result_for_plot = gr.State()
    true_lift_state = gr.State()
    roas_est_state = gr.State()
    sem_data_state = gr.State()
    sem_data_head_state = gr.State()
    hidden_lift = gr.State()
    hidden_start_date = gr.State()
    sem_weights_state = gr.State()
    best_lever_state = gr.State()
    target_state = gr.State() # Holds the locked target
    mm_results_table_state = gr.State()


    gr.Markdown("# 🗺️ GeoLift Data Preview")

    with gr.Tabs():

        # --- TAB 1: SEM ---
        with gr.Tab("Pre-Test Design", id="feature_tab"):
            with gr.Row(): # 📝 Same trick here!
              with gr.Column(scale=1, min_width=300):
                gr.Markdown("### 1. File Upload")
                sem_file_input = gr.File(label="Upload CSV or Excel", file_types=[".csv", ".xlsx", ".xls"])

                gr.Markdown("### 2. Auto Causal Discovery (LiNGAM)")

                target = gr.Dropdown(choices=[], label="Target Variable (Y)")

                # NEW: Dropdown for Date and Geo
                drop_cols = gr.Dropdown(
                    choices=[],
                    multiselect=True,
                    label="Columns to Drop (e.g., Date, Geo)"
                )

                auto_causal_btn = gr.Button("Run Auto Causal Discovery", variant="primary")

                gr.Markdown("### 3. SEM Relationship Builder")

                category_note = gr.Textbox(label="Category / Note (Optional)", placeholder="e.g., Funnel Mechanics")

                relationship_type = gr.Radio(
                    choices=["Regression (Drives)", "Covariance (Shared Variance)"],
                    value="Regression (Drives)",
                    label="Relationship Type"
                )


                drivers = gr.Dropdown(
                    choices=[],
                    multiselect=True,
                    label="Drivers / Correlates (X)"
                )

                add_btn = gr.Button("Add Rule to Model", variant="primary")


               # --- RIGHT COLUMN: The Output ---
              with gr.Column(scale=5):
                  gr.Markdown("### Discovered Causal Graph")
                  #Plot component for Matplotlib
                  causal_plot_output = gr.Plot(label="DirectLiNGAM Graph")

                  gr.Markdown("### Generated `semopy` Script")

                  # This holds the accumulated string
                  model_display = gr.Code(label="Model Syntax", language="python", lines=15)

                  with gr.Row():
                      clear_btn = gr.Button("Clear All")
                      undo_btn = gr.Button("Undo Last Rule")
                      run_btn = gr.Button("Fit SEM Model", variant="stop")

                    # --- Results Section (Below the Columns) ---
                  with gr.Row():
                      with gr.Column(scale=2):
                          # Output 1: The Table
                          # results_output = gr.HTML(label="SEM Results")
                          results_output = gr.Dataframe(
                                label="SEM Statistical Results",
                                interactive=False, # Keeps it as a display-only table
                                wrap=True,         # Ensures long variable names don't get cut off
                                max_height=500     # Adds a scrollbar if the model is huge
                            )

                          # Output 2: The Plot
                          # Using type="filepath" allows Gradio to read the image we just saved
                          plot_output = gr.Image(label="Path Diagram", type="pil")

                  with gr.Row():
                    sem_weights = gr.Textbox(label="SEM Weights", placeholder="{'avg_revenue': 1.0}", interactive=True, lines=6)

                  with gr.Row():
                    explain_btn = gr.Button("✨ Finalize Results", variant="primary")

                  with gr.Row():
                    sem_output_md = gr.Markdown(visible=False)



        # --- TAB 2: DESIGN ---
        # with gr.Tab("Design", id="design_tab"):
        with gr.Tab("Pre-Test Planning", id="design_tab", visible=False) as design_tab_view:
          with gr.Row(): # 📝 Wrap the whole tab content in a Row

            # with gr.Sidebar():
          # 📝 This Column acts as your Sidebar (scale=1 keeps it narrow)
            with gr.Column(scale=1, min_width=300):
                # file_input = gr.File(label="Upload CSV or Excel", file_types=[".csv", ".xlsx", ".xls"])
                # target_column = gr.Dropdown(label="Select Target Column (KPI)", choices=[])
                target_column = gr.Dropdown(label="Target Column (KPI)", choices=[])
                cost_column = gr.Dropdown(label="Select Cost Column", choices=[])
                date_column = gr.Dropdown(label="Select Date Column", choices=[])
                additional_cols_cluster = gr.Dropdown(label="Select Attributes For Cluster", choices=[],  multiselect=True)
                exclude_dates = gr.Dropdown(label="Exclude Specific Dates (Holidays/Events)", choices=[], multiselect=True, interactive=False)
                geo_column = gr.Dropdown(label="Select Geo Column", choices=[])

                run_button = gr.Button("Run Cluster Outlier Analysis", variant="primary", elem_id='run_button')

                test_direction = gr.Radio(
                    label="Test Direction",
                    choices=["Scale-up", "Hold-out"],
                    value="Scale-up",
                    visible=False
                )

                target_type = gr.Radio(
                    label="Is the Target Revenue?",
                    choices=["Revenue", "Conversions (CPA)"],
                    value="Revenue",
                    visible=False
                )

                minimum_detectable_iROAS = gr.Number(
                    label="Minimum detectable iROAS",
                    value=3.0,
                    visible=False
                )

                experiment_date = gr.Number(
                    label="Experiment Length (Days)",
                    value=8,
                    visible=False
                )

                cluster_filter = gr.Dropdown(
                    label="Filter by Cluster",
                    choices=[],
                    multiselect=True,
                    visible=False
                )

                max_allowed_cost = gr.Number(label="Maximum Allowed Budget Cost", value=350000, visible=False, interactive=False)

                planned_budget = gr.Number(
                    label="Budget to Add",
                    value=15000,
                    visible=False
                )

                required_test_geo_arm = gr.Dropdown(
                    label="Required Treatment Geos",
                    choices=[],
                    multiselect=True,
                    visible=False
                )

                exclude_from_test_geo_arm = gr.Dropdown(
                    label="Exclude from Treatment",
                    choices=[],
                    multiselect=True,
                    visible=False
                )

                exclude_completely = gr.Dropdown(
                    label="Exclude Completely",
                    choices=[],
                    multiselect=True,
                    visible=False
                )

                mamtchmarket_button = gr.Button("Match Market Analysis", variant="primary", elem_id='mm_button', visible=False)

                match_selector = gr.Dropdown(label="Select Match to Evaluate", choices=[], visible=False)

                summary_button = gr.Button("Generate Design Summary", variant="secondary", visible=False)


            with gr.Column(scale=5):
                gr.Markdown("### 📝 Design Plan")
                status_label = gr.Markdown("Upload data to begin.")
                data_view = gr.Dataframe(label="Dataset Preview (First 5 rows)")
                cluster_plot = gr.Plot(label="Geo Clusters")
                mm_results_table = gr.Dataframe(label="Matched Market Designs")
                summary_output_text = gr.Markdown(visible=False)
                ci_plot_output = gr.Plot(label="CausalImpact Counterfactual Fit", visible=False)
                tri_plot_output = gr.Plot(label="Triangulated Ensemble Bounds", visible=False)

        # --- TAB 3: RESULT ANALYSIS ---
        with gr.Tab("Post-Planning Result Analysis", id="analyze_tab"):
            with gr.Row(): # 📝 Same trick here!

                # 📝 Simulated Sidebar for the Analyze Tab
                with gr.Column(scale=1, min_width=300):
                    # Be careful not to use the exact same variable name 'file_input'
                    # as Tab 1 if you want them to be separate components!
                    analyze_file_input = gr.File(label="Upload CSV or Excel", file_types=[".csv", ".xlsx", ".xls"])
                    target_column_test = gr.Dropdown(label="Select Target Column (KPI)", choices=[])
                    cost_column_test = gr.Dropdown(label="Select Cost Column", choices=[])
                    date_column_test = gr.Dropdown(label="Select Date Column", choices=[])
                    geo_column_test = gr.Dropdown(label="Select Geo Column", choices=[])
                    geo_selector = gr.Dropdown(label="Select Geos to Plot (Leave blank for Top 12)", multiselect=True, choices=[], interactive=True)
                    period_column = gr.Dropdown(label="Select Period Column", choices=[])
                    test_start = gr.DateTime(label="Pre-Test Start Date",  include_time=False, type="string")
                    test_end = gr.DateTime(label="Experiment Start Date", include_time=False, type="string")

                    bulk_paste_box = gr.Textbox(label="Paste Geos (Comma Separated)", placeholder="Paste here and press Enter...", interactive=True)

                    test_group_geo = gr.Dropdown(label="Select Test Group Geo", multiselect=True, choices=[], interactive=True, allow_custom_value=True)


                    excluded_cntrl_geo = gr.Dropdown(label="Exclude from Control", multiselect=True, choices=[], interactive=True)


                    use_cooldown_checkbox = gr.Checkbox(label="⏳ Enable Cooldown Period Analysis", value=False)

                    test_analyze_button = gr.Button("Generate Pre-Test & Test Result", visible=True, variant="primary")

                    # gr.Markdown("### 📍 Directional Geo-Level Contribution")
                    # gr.Markdown("*Note: Individual market estimates are allocated proportionally based on historical baseline share to preserve the statistical integrity of the aggregate causal model.*")

                    # geo_breakdown_btn = gr.Button("Calculate Geo-Level Lift", variant="secondary")


                # 📝 Main Content for Analyze Tab
                with gr.Column(scale=5):
                    gr.Markdown("### 📊 Analysis Results")
                    data_view_test = gr.Dataframe(label="Dataset Preview (First 5 rows)")
                    geo_facet_plot = gr.Plot(label="Geo Revenue Time Series")
                    diagnostic_summary_text = gr.Markdown(visible=False)
                    waterfall_plot_output = gr.Plot(label="Executive Summary: Total Revenue Impact")
                    causal_impact_plot_output = gr.Plot(label="Interactive Causal Impact Visualization")

                    # gr.Markdown("### 📍 Directional Geo-Level Contribution")
                    # gr.Markdown("*Note: Individual market estimates are allocated proportionally based on historical baseline share to preserve the statistical integrity of the aggregate causal model.*")
                    # geo_breakdown_df = gr.Dataframe(label="Fair-Share Lift Distribution", interactive=False, wrap=True)



    # ==========================================
    # 4. EVENT TRIGGERS
    # ==========================================

    target_type.change(
        fn=update_target_label,
        inputs=target_type,
        outputs=minimum_detectable_iROAS
    )

    test_direction.change(
        fn=update_budget_label,
        inputs=test_direction,
        outputs=planned_budget
    )


    # Tab 3: RESULT ANALYSIS VARIABLE
    analyze_file_input.upload(
        fn=process_file_and_preview_test,
        inputs=[analyze_file_input],
        outputs=[test_data_state, data_view_test, target_column_test, cost_column_test, date_column_test, geo_column_test, period_column, test_start, test_end]
    )

    sem_file_input.upload(
        fn=sem_process_file_and_preview_test,
        inputs=[sem_file_input],
        outputs=[
            sem_data_state,
            sem_data_head_state,
            category_note,
            relationship_type,
            target,
            drivers,
            drop_cols
        ]
    )


    run_button.click(
        fn=run_clustering,
        inputs=[
            sem_data_state,
            target_column,
            cost_column,
            additional_cols_cluster,
            date_column,
            geo_column,
            exclude_dates,
            sem_weights
        ],
        outputs=[
            status_label, cluster_plot,
            cluster_filter, test_direction, target_type, minimum_detectable_iROAS,
            experiment_date, max_allowed_cost, planned_budget,
            required_test_geo_arm, exclude_from_test_geo_arm, exclude_completely,
            mamtchmarket_button, clustered_data_state
        ]
    )

    cluster_filter.change(
        fn=update_geo_dropdowns,
        inputs=[clustered_data_state, cluster_filter, geo_column],
        outputs=[required_test_geo_arm, exclude_from_test_geo_arm, exclude_completely]
    )

    mamtchmarket_button.click(
        fn=run_match_market,
        inputs=[
            clustered_data_state, cluster_filter, target_column, geo_column, date_column,
            target_type, test_direction, minimum_detectable_iROAS,
            experiment_date, max_allowed_cost, planned_budget,
            required_test_geo_arm, exclude_from_test_geo_arm, exclude_completely
        ],
        outputs=[status_label, mm_results_table_state, geolist_state,               # 3. INVISIBLE: Stores df_geolist
            matched_designs_state,       # 4. INVISIBLE: Stores matched_designs
            optimal_testing_data_state,  # 5. INVISIBLE: Stores optimal_testing_data
            match_selector,              # 6. Catches the Dropdown update
            summary_button,
            mm_results_table]
    )

# 📝 Directly triggers your heavy-lifting function!
# 📝 Trigger for the Summary Button
    summary_button.click(
        fn=design_summary_choosen_design,
        inputs=[
            match_selector,              # 1. e.g., "Match 0"
            matched_designs_state,       # 2. matched_designs
            optimal_testing_data_state,  # 3. ONLY the data for the chosen cluster
            experiment_date,             # 4. experiment duration
            target_column,               # 5. target column
            geo_column,                  # 6. geo column
            date_column,                 # 7. date column
            cost_column,                 # 8. cost column
            minimum_detectable_iROAS,    # 9. iROAS / CPA
            test_direction,              # 10. Hold-out or Scale-up
            target_type,                 # 11. Revenue vs Conversions
            planned_budget,              # 12. budget_cut
            mm_results_table_state       # 13. The visible Gradio Dataframe from run_match_market
        ],
        outputs=[summary_output_text, ci_plot_output, tri_plot_output]    # Outputs to the Markdown component
    )

    # Tab 3: RESULT ANALYSIS VARIABLE
    date_column_test.change(
        fn=pre_test_start_date,
        inputs=[test_data_state, date_column_test, period_column],
        outputs=[test_start, test_end]
    )



    # Tab 3: RESULT ANALYSIS VARIABLE
    period_column.change(
        fn=pre_test_start_date,
        inputs=[test_data_state, date_column_test, period_column],
        outputs=[test_start, test_end]
    )

    # Tab 3: RESULT ANALYSIS VARIABLE
    test_analyze_button.click(
            fn=generate_geo_facet_plot,
            inputs=[
                test_data_state,     # 1. The full, invisible dataset
                geo_column_test,     # 2. The geo column dropdown
                date_column_test,    # 3. The date column dropdown
                target_column_test,   # 4. The target (revenue) dropdown
                geo_selector
            ],
            outputs=[geo_facet_plot] # 📝 Spits the Plotly figure out to the UI!
    )

    # Tab 3: RESULT ANALYSIS VARIABLE
    bulk_paste_box.submit(
        fn=apply_geo_bulk_paste,
        inputs=[bulk_paste_box, test_group_geo],
        outputs=[test_group_geo, bulk_paste_box] # Updates dropdown, clears textbox
    )

    # Tab 3: RESULT ANALYSIS VARIABLE
    tbr_math_event = test_analyze_button.click(
        fn=diagnostic_summary,
        inputs=[
            test_data_state,       # df
            geo_column_test,       # geo_col
            target_column_test,    # target_col
            cost_column_test,      # cost_col
            period_column,         # period_col
            date_column_test,      # date_col
            test_group_geo,        # treatment_geos
            excluded_cntrl_geo,    # excluded_control_geo
            use_cooldown_checkbox  #True/False toggle
        ],
        outputs=[result_for_plot, true_lift_state, roas_est_state,
                 diagnostic_summary_text] # 📝 Spits the Markdown out to the UI!
    )

    # Tab 3: RESULT ANALYSIS VARIABLE
    tbr_math_event.then(
        fn=diagnostic_plots,
        inputs=[
            result_for_plot,
            target_column_test,
            date_column_test,
            cost_column_test,
            true_lift_state,
            roas_est_state
        ],
        outputs=[waterfall_plot_output, causal_impact_plot_output]
    )

    # Tab 3: RESULT ANALYSIS VARIABLE
    geo_column_test.change(
        fn=populate_geo_choices,
        inputs=[test_data_state, geo_column_test],
        outputs=[geo_selector, test_group_geo, excluded_cntrl_geo]
    )

    # --- SEM Interactions ---
    auto_causal_btn.click(
            fn=run_auto_causal,
            inputs=[sem_data_state, target, drop_cols],
            outputs=[causal_plot_output, model_display]
        )

    add_btn.click(
        fn=add_rule,
        inputs=[model_display, relationship_type, target, drivers, category_note],
        outputs=[model_display]
    )

    # This button takes the results of the SEM model and sends them to Gemini
    explain_btn.click(
        fn=explain_sem_results,
        inputs=[results_output, target], # ✅ Passes both the statistical table and your Target column
        outputs=[sem_output_md, best_lever_state, target_state] # ✅ Captures the report, best lever, and target variable
    ).then(
        fn=lambda: gr.update(visible=True),
        inputs=None,
        outputs=[design_tab_view]
    )

    # Update the Tab Selection Trigger
    design_tab_view.select(
        fn=prep_design_tab,
        # ✅ Feed the best_lever_state into the prep function
        inputs=[sem_data_state, target_state, best_lever_state, sem_weights, drop_cols],
        outputs=[data_view, target_column, cost_column, date_column, geo_column, additional_cols_cluster]
    )

    clear_btn.click(fn=clear_model, inputs=[], outputs=[model_display, results_output, plot_output])

    # In a real scenario, pass your DataFrame 'df' instead of None
    run_btn.click(fn=run_sem, inputs=[model_display, sem_data_state, target], outputs=[results_output, sem_weights, plot_output])

    undo_btn.click(
        fn=undo_last_rule,
        inputs=[model_display],
        outputs=[model_display]
    )

    # # Tab 3: RESULT ANALYSIS VARIABLE
    # geo_breakdown_btn.click(
    #     fn=generate_geo_level_report,
    #     inputs=[
    #         test_data_state,      # Your full dataframe
    #         test_group_geo,     # Dropdown or list of treatment cities
    #         geo_column_test,
    #         target_column_test,
    #         cost_column_test,
    #         date_column_test,
    #         true_lift_state,             # Captured from your main analysis
    #         test_end        # Captured from your main analysis
    #     ],
    #     outputs=[geo_breakdown_df]
    # )

clear_output()

# Deployment

In [8]:
# @title Deployment App
if __name__ == "__main__":
# Fix the ipykernel bug by adding the missing attribute that the close method looks for
    if not hasattr(OutStream, "watch_fd_thread"):
        OutStream.watch_fd_thread = None
    try:
      with suppress_stdout_stderr():
          # output.eval_js('google.colab.output.setIframeHeight("15000")') # Set to 800px
          # The app launches, the URL is printed to the void, and the plots stay alive
          app.launch(height=1200, quiet=True, show_api=False, share=True)
    except:
      app.launch(height=1200, quiet=True, show_api=False, share=True)
      clear_output()

      with suppress_stdout_stderr():
          # output.eval_js('google.colab.output.setIframeHeight("15000")') # Set to 800px
          # The app launches, the URL is printed to the void, and the plots stay alive
          app.launch(height=1200, quiet=True, show_api=False, share=True)
